[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/03_linear_systems_and_direct_factorizations/exercises.ipynb)

# Module 03 — Exercises: Linear Systems and Direct Factorizations

Forty solved problems in four tiers. Every problem carries a statement, a one-line intuition, a
stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): norms are written $\lVert x \rVert$,
transposes $A^\top$, and $u = 2^{-53}$ is the unit roundoff.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

import scipy.linalg as sla

EPS = np.finfo(float).eps
U_ROUND = EPS / 2
print(f"machine epsilon = {EPS:.4e}   unit roundoff u = {U_ROUND:.4e}")

machine epsilon = 2.2204e-16   unit roundoff u = 1.1102e-16


## L0 — Concept Checks

### Problem L0.1 — Back substitution

**Statement.** Solve $Ux = b$ with

$$
U = \begin{bmatrix} 2 & 1 & -1 \\ 0 & 3 & 2 \\ 0 & 0 & 4 \end{bmatrix},
\qquad b = \begin{bmatrix} 3 \\ 10 \\ 8 \end{bmatrix} .
$$

**Intuition.** The last equation involves one unknown, so the system unwinds from the bottom.

**Solution.**

*Step 1.* $4x_3 = 8$, so $x_3 = 2$.

*Step 2.* $3x_2 + 2(2) = 10$, so $x_2 = 2$.

*Step 3.* $2x_1 + 2 - 2 = 3$, so $x_1 = 3/2$.

$$
\boxed{x = \left(\tfrac32,\; 2,\; 2\right)^\top}
$$

**Key takeaway.** Back substitution costs $n^2$ flops, against $\tfrac23 n^3$ for the
elimination that produced $U$ — which is why one factors once and solves many times.

In [2]:
U = np.array([[2.0, 1.0, -1.0], [0.0, 3.0, 2.0], [0.0, 0.0, 4.0]])
b = np.array([3.0, 10.0, 8.0])
x = sla.solve_triangular(U, b, lower=False)
print("x =", x, "  residual =", np.linalg.norm(U @ x - b))
assert np.allclose(x, [1.5, 2.0, 2.0])

x = [1.5 2.  2. ]   residual = 0.0


### Problem L0.2 — Forward substitution

**Statement.** Solve $Ly = b$ with

$$
L = \begin{bmatrix} 1 & 0 & 0 \\ 2 & 1 & 0 \\ -1 & 3 & 1 \end{bmatrix},
\qquad b = \begin{bmatrix} 4 \\ 11 \\ 15 \end{bmatrix} .
$$

**Intuition.** The first equation involves one unknown, so the system unwinds from the top.

**Solution.**

*Step 1.* $y_1 = 4$.

*Step 2.* $2(4) + y_2 = 11$, so $y_2 = 3$.

*Step 3.* $-4 + 3(3) + y_3 = 15$, so $y_3 = 10$.

$$
\boxed{y = (4,\; 3,\; 10)^\top}
$$

**Key takeaway.** Forward substitution on a unit lower triangular matrix needs no divisions at
all, since every diagonal entry is $1$.

In [3]:
L = np.array([[1.0, 0.0, 0.0], [2.0, 1.0, 0.0], [-1.0, 3.0, 1.0]])
b = np.array([4.0, 11.0, 15.0])
y = sla.solve_triangular(L, b, lower=True)
print("y =", y, "  residual =", np.linalg.norm(L @ y - b))
assert np.allclose(y, [4.0, 3.0, 10.0])

y = [ 4.  3. 10.]   residual = 0.0


### Problem L0.3 — An elimination matrix and its inverse

**Statement.** Write the matrix $E_{21} \in \mathbb{R}^{3 \times 3}$ that performs
$R_2 \leftarrow R_2 - 3R_1$, and give $E_{21}^{-1}$.

**Intuition.** The inverse of "subtract three times row 1" is "add three times row 1".

**Solution.**

*Step 1.* Apply the operation to $I_3$:

$$
E_{21} = \begin{bmatrix} 1 & 0 & 0 \\ -3 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix} .
$$

*Step 2.* Flip the sign of the single off-diagonal entry:

$$
E_{21}^{-1} = \begin{bmatrix} 1 & 0 & 0 \\ 3 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix} .
$$

$$
\boxed{E_{21} = I - 3e_2e_1^\top, \qquad E_{21}^{-1} = I + 3e_2e_1^\top}
$$

**Key takeaway.** $(e_2e_1^\top)^2 = 0$, so $(I - 3e_2e_1^\top)(I + 3e_2e_1^\top) = I$. This is
exactly why the multipliers in Gaussian elimination can be collected into $L$ with their signs
flipped and no further arithmetic.

In [4]:
e1 = np.eye(3)[:, 0]
e2 = np.eye(3)[:, 1]
E = np.eye(3) - 3 * np.outer(e2, e1)
Einv = np.eye(3) + 3 * np.outer(e2, e1)
print("E =\n", E, "\nE^-1 =\n", Einv, "\nE E^-1 =\n", E @ Einv)
assert np.allclose(E @ Einv, np.eye(3))
assert np.allclose(np.linalg.inv(E), Einv)

E =
 [[ 1.  0.  0.]
 [-3.  1.  0.]
 [ 0.  0.  1.]] 
E^-1 =
 [[1. 0. 0.]
 [3. 1. 0.]
 [0. 0. 1.]] 
E E^-1 =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


### Problem L0.4 — Gaussian elimination on a $2 \times 2$

**Statement.** Solve $Ax = b$ with
$A = \left[\begin{smallmatrix}3 & 2 \\ 6 & 7\end{smallmatrix}\right]$ and
$b = \left(8, 19\right)^\top$.

**Intuition.** One elimination step makes the system triangular.

**Solution.**

*Step 1.* The multiplier is $l_{21} = 6/3 = 2$, and $R_2 \leftarrow R_2 - 2R_1$ gives

$$
\begin{bmatrix} 3 & 2 \\ 0 & 3 \end{bmatrix} x = \begin{bmatrix} 8 \\ 3 \end{bmatrix} .
$$

*Step 2.* Back substitution: $x_2 = 1$, then $3x_1 + 2 = 8$, so $x_1 = 2$.

$$
\boxed{x = (2,\; 1)^\top}
$$

**Key takeaway.** The two pivots are $3$ and $3$; their product $9$ is $\det A = 21 - 12$.

In [5]:
A = np.array([[3.0, 2.0], [6.0, 7.0]])
b = np.array([8.0, 19.0])
x = np.linalg.solve(A, b)
print("x =", x, "  pivots 3, 3 -> product", 3 * 3, "  det A =", np.linalg.det(A))
assert np.allclose(x, [2.0, 1.0])
assert np.isclose(np.linalg.det(A), 9.0)

x = [2. 1.]   pivots 3, 3 -> product 9   det A = 8.999999999999998


### Problem L0.5 — Permutation matrices are orthogonal

**Statement.** For
$P = \left[\begin{smallmatrix}0&1&0\\0&0&1\\1&0&0\end{smallmatrix}\right]$, describe $PA$ and
show $P^\top P = I_3$.

**Intuition.** Row $i$ of $P$ is a standard basis vector, so row $i$ of $PA$ is one row of $A$.

**Solution.**

*Step 1.* With $A$ written by rows $r_1^\top, r_2^\top, r_3^\top$,

$$
PA = \begin{bmatrix} r_2^\top \\ r_3^\top \\ r_1^\top \end{bmatrix} .
$$

*Step 2.* The columns of $P$ are $e_3, e_1, e_2$, an orthonormal set, so
$(P^\top P)_{ij} = e_{\sigma(i)}^\top e_{\sigma(j)} = \delta_{ij}$.

$$
\boxed{PA \text{ permutes the rows of } A, \qquad P^{-1} = P^\top}
$$

**Key takeaway.** Because $P^{-1} = P^\top$, the row swaps recorded in $PA = LU$ are undone by a
transpose, at no arithmetic cost and with no effect on conditioning:
$\kappa_2(PA) = \kappa_2(A)$.

In [6]:
P = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]])
A = rng.standard_normal((3, 3))
print("P^T P - I =\n", P.T @ P - np.eye(3))
print("rows of PA equal rows 2,3,1 of A:", np.allclose(P @ A, A[[1, 2, 0], :]))
print("kappa_2(PA) =", np.linalg.cond(P @ A), " kappa_2(A) =", np.linalg.cond(A))
assert np.allclose(P.T @ P, np.eye(3))
assert np.allclose(P @ A, A[[1, 2, 0], :])
assert np.isclose(np.linalg.cond(P @ A), np.linalg.cond(A))

P^T P - I =
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
rows of PA equal rows 2,3,1 of A: True
kappa_2(PA) = 5.519171215307488  kappa_2(A) = 5.519171215307491


### Problem L0.6 — A matrix with no LU factorization

**Statement.** Explain why $A = \left[\begin{smallmatrix}0 & 2 \\ 3 & 4\end{smallmatrix}\right]$
has no factorization $A = LU$ with $L$ unit lower triangular, and give $P, L, U$ with
$PA = LU$.

**Intuition.** The first multiplier would be $3/0$.

**Solution.**

*Step 1.* $A_1 = [0]$ is singular, so Theorem 4.4 forbids an LU factorization of the
nonsingular matrix $A$.

*Step 2.* Swap the rows:

$$
P = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix},
\qquad
PA = \begin{bmatrix} 3 & 4 \\ 0 & 2 \end{bmatrix},
$$

which is already upper triangular, so $L = I_2$ and $U = PA$.

$$
\boxed{P = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}, \quad L = I_2, \quad U = \begin{bmatrix} 3 & 4 \\ 0 & 2 \end{bmatrix}}
$$

**Key takeaway.** Theorem 4.5 guarantees $PA = LU$ for **every** square matrix; only the
unpermuted version can fail.

In [7]:
A = np.array([[0.0, 2.0], [3.0, 4.0]])
P = np.array([[0.0, 1.0], [1.0, 0.0]])
L = np.eye(2)
U = np.array([[3.0, 4.0], [0.0, 2.0]])
print("A_1 = [0], singular:", np.isclose(A[0, 0], 0.0))
print("||P A - L U||_F =", np.linalg.norm(P @ A - L @ U))
Ps, Ls, Us = sla.lu(A)
print("scipy lu: ||P^T A - L U|| =", np.linalg.norm(Ps.T @ A - Ls @ Us))
assert np.allclose(P @ A, L @ U)

A_1 = [0], singular: True
||P A - L U||_F = 0.0
scipy lu: ||P^T A - L U|| = 0.0


### Problem L0.7 — Cholesky breakdown detects indefiniteness

**Statement.** Decide whether $A = \left[\begin{smallmatrix}1 & 2 \\ 2 & 3\end{smallmatrix}\right]$
is positive definite by attempting its Cholesky factorization.

**Intuition.** Cholesky takes a square root at every diagonal step; a negative radicand is the
failure certificate.

**Solution.**

*Step 1.* $l_{11} = \sqrt{1} = 1$ and $l_{21} = 2/1 = 2$.

*Step 2.* $l_{22}^2 = 3 - 2^2 = -1 \lt 0$, so no real $l_{22}$ exists.

*Step 3.* The failing pivot points at a direction of negative curvature: with
$x = (-2, 1)^\top$,

$$
x^\top A x = -1 \lt 0 .
$$

$$
\boxed{\text{Not positive definite: the second pivot is } -1}
$$

**Key takeaway.** By Theorem 4.6 the existence of the Cholesky factor is equivalent to positive
definiteness, so a failed factorization is a proof, not an inconvenience.

In [8]:
A = np.array([[1.0, 2.0], [2.0, 3.0]])
try:
    np.linalg.cholesky(A)
    print("unexpectedly succeeded")
except np.linalg.LinAlgError as err:
    print("cholesky failed:", err)
xneg = np.array([-2.0, 1.0])
print("second pivot =", 3 - 2 ** 2, "   x^T A x at (-2,1) =", xneg @ A @ xneg)
print("eigenvalues  =", np.linalg.eigvalsh(A))
assert xneg @ A @ xneg < 0
assert np.linalg.eigvalsh(A)[0] < 0

cholesky failed: Matrix is not positive definite
second pivot = -1    x^T A x at (-2,1) = -1.0
eigenvalues  = [-0.2361  4.2361]


### Problem L0.8 — Cholesky of a $2 \times 2$ kernel matrix

**Statement.** Give the Cholesky factor of
$K = \left[\begin{smallmatrix}1 & e^{-1} \\ e^{-1} & 1\end{smallmatrix}\right]$.

**Intuition.** A squared-exponential kernel matrix at two distinct points has unit diagonal and
off-diagonal strictly between $0$ and $1$.

**Solution.**

*Step 1.* $l_{11} = \sqrt{1} = 1$.

*Step 2.* $l_{21} = e^{-1}/1 = e^{-1}$.

*Step 3.* $l_{22} = \sqrt{1 - e^{-2}}$, which is real and positive because $e^{-1} \lt 1$.

$$
\boxed{L = \begin{bmatrix} 1 & 0 \\ e^{-1} & \sqrt{1 - e^{-2}} \end{bmatrix}}
$$

**Key takeaway.** Distinct sample points give $\lvert k_{12} \rvert \lt 1$ and hence a positive
definite $K$; coincident points make $K$ singular, which is why a jitter term $\sigma^2 I$ is
standard practice.

In [9]:
e_inv = np.exp(-1.0)
K = np.array([[1.0, e_inv], [e_inv, 1.0]])
L_hand = np.array([[1.0, 0.0], [e_inv, np.sqrt(1 - e_inv ** 2)]])
print("hand L =\n", L_hand, "\nnumpy  =\n", np.linalg.cholesky(K))
print("||K - L L^T||_F =", np.linalg.norm(K - L_hand @ L_hand.T))
Ksing = np.array([[1.0, 1.0], [1.0, 1.0]])
print("coincident points: eigenvalues", np.linalg.eigvalsh(Ksing), "-> singular")
assert np.allclose(np.linalg.cholesky(K), L_hand)
assert np.isclose(np.linalg.det(Ksing), 0.0)

hand L =
 [[1.     0.    ]
 [0.3679 0.9299]] 
numpy  =
 [[1.     0.    ]
 [0.3679 0.9299]]
||K - L L^T||_F = 1.1102230246251565e-16
coincident points: eigenvalues [0. 2.] -> singular


## L1 — Foundations

### Problem L1.1 — LU of a $3 \times 3$

**Statement.** Compute the LU factorization, without pivoting, of

$$
A = \begin{bmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{bmatrix} .
$$

**Intuition.** $U$ records where elimination ends; $L$ records the multipliers it used to get
there.

**Solution.**

*Step 1 — check the hypothesis of Theorem 4.4.* $A_1 = [2]$ and
$A_2 = \left[\begin{smallmatrix}2 & 1 \\ 4 & 3\end{smallmatrix}\right]$ are nonsingular, so a
unique LU factorization exists.

*Step 2 — clear column 1.* $l_{21} = 4/2 = 2$ gives $[0,1,1]$; $l_{31} = 8/2 = 4$ gives
$[0,3,5]$.

*Step 3 — clear column 2.* $l_{32} = 3/1 = 3$ gives $[0,0,2]$.

$$
\boxed{L = \begin{bmatrix} 1 & 0 & 0 \\ 2 & 1 & 0 \\ 4 & 3 & 1 \end{bmatrix}, \qquad
U = \begin{bmatrix} 2 & 1 & 1 \\ 0 & 1 & 1 \\ 0 & 0 & 2 \end{bmatrix}}
$$

**Key takeaway.** The multipliers are stored, not discarded: that is the entire difference
between "Gaussian elimination" and "LU factorization".

In [10]:
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L = np.array([[1.0, 0.0, 0.0], [2.0, 1.0, 0.0], [4.0, 3.0, 1.0]])
U = np.array([[2.0, 1.0, 1.0], [0.0, 1.0, 1.0], [0.0, 0.0, 2.0]])
print("||A - L U||_F =", np.linalg.norm(A - L @ U))
print("leading submatrix determinants:", [np.linalg.det(A[:k, :k]) for k in (1, 2, 3)])
assert np.allclose(A, L @ U)

||A - L U||_F = 0.0
leading submatrix determinants: [np.float64(2.0), np.float64(2.0), np.float64(4.000000000000002)]


### Problem L1.2 — Solving with the LU factors

**Statement.** Using $L$ and $U$ from Problem L1.1, solve $Ax = b$ for
$b = (4, 10, 26)^\top$.

**Intuition.** $LUx = b$ splits into two triangular solves.

**Solution.**

*Step 1 — forward solve $Ly = b$.* $y_1 = 4$; $2(4) + y_2 = 10$ gives $y_2 = 2$;
$4(4) + 3(2) + y_3 = 26$ gives $y_3 = 4$.

*Step 2 — back solve $Ux = y$.* $2x_3 = 4$ gives $x_3 = 2$; $x_2 + 2 = 2$ gives $x_2 = 0$;
$2x_1 + 0 + 2 = 4$ gives $x_1 = 1$.

$$
\boxed{x = (1,\; 0,\; 2)^\top}
$$

**Key takeaway.** Once $L$ and $U$ are in hand, each new right-hand side costs $2n^2$ flops
rather than $\tfrac23 n^3$.

In [11]:
b = np.array([4.0, 10.0, 26.0])
y = sla.solve_triangular(L, b, lower=True)
x = sla.solve_triangular(U, y, lower=False)
print("y =", y, "  x =", x, "  residual =", np.linalg.norm(A @ x - b))
assert np.allclose(y, [4.0, 2.0, 4.0])
assert np.allclose(x, [1.0, 0.0, 2.0])

y = [4. 2. 4.]   x = [1. 0. 2.]   residual = 0.0


### Problem L1.3 — Separating scale from shear: $A = LDU_1$

**Statement.** Factor $A = \left[\begin{smallmatrix}2 & 4 \\ 6 & 17\end{smallmatrix}\right]$ as
$LDU_1$ with $L$ unit lower triangular, $D$ diagonal and $U_1$ unit upper triangular.

**Intuition.** Pulling the pivots out of $U$ leaves a matrix with ones on the diagonal, so the
scaling lives in one factor and the shears in the other two.

**Solution.**

*Step 1 — eliminate.* $l_{21} = 6/2 = 3$, giving
$U = \left[\begin{smallmatrix}2 & 4 \\ 0 & 5\end{smallmatrix}\right]$.

*Step 2 — extract the diagonal.* $D = \operatorname{diag}(2, 5)$ and $U_1 = D^{-1}U$.

$$
\boxed{L = \begin{bmatrix} 1 & 0 \\ 3 & 1 \end{bmatrix}, \quad
D = \begin{bmatrix} 2 & 0 \\ 0 & 5 \end{bmatrix}, \quad
U_1 = \begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix}}
$$

**Key takeaway.** $LDU_1$ is the unsymmetric ancestor of $LDL^\top$: when $A$ is symmetric,
$U_1 = L^\top$ and the two factorizations coincide.

In [12]:
A = np.array([[2.0, 4.0], [6.0, 17.0]])
L = np.array([[1.0, 0.0], [3.0, 1.0]])
D = np.diag([2.0, 5.0])
U1 = np.array([[1.0, 2.0], [0.0, 1.0]])
print("||A - L D U1||_F =", np.linalg.norm(A - L @ D @ U1))
Asym = np.array([[2.0, 4.0], [4.0, 13.0]])
Ls = np.array([[1.0, 0.0], [2.0, 1.0]])
Ds = np.diag([2.0, 5.0])
print("symmetric case: ||Asym - L D L^T||_F =", np.linalg.norm(Asym - Ls @ Ds @ Ls.T))
assert np.allclose(A, L @ D @ U1)
assert np.allclose(Asym, Ls @ Ds @ Ls.T)

||A - L D U1||_F = 0.0
symmetric case: ||Asym - L D L^T||_F = 0.0


### Problem L1.4 — One step of partial pivoting

**Statement.** For

$$
A = \begin{bmatrix} 1 & 4 & 2 \\ 3 & 2 & 1 \\ 2 & 1 & 5 \end{bmatrix},
$$

choose the pivot row for column $1$ under partial pivoting and carry out the first elimination
step.

**Intuition.** Picking the largest entry in the column keeps every multiplier at most $1$ in
modulus, which is what Theorem 4.5 needs.

**Solution.**

*Step 1 — choose.* The column-$1$ magnitudes are $1, 3, 2$, so row $2$ is the pivot row. Swap
rows $1$ and $2$:

$$
P_1 A = \begin{bmatrix} 3 & 2 & 1 \\ 1 & 4 & 2 \\ 2 & 1 & 5 \end{bmatrix} .
$$

*Step 2 — multipliers.* $l_{21} = \tfrac13$ and $l_{31} = \tfrac23$, both at most $1$.

*Step 3 — eliminate.*

$$
R_2 \leftarrow R_2 - \tfrac13 R_1 = \left[0,\; 4 - \tfrac23,\; 2 - \tfrac13\right]
= \left[0,\; \tfrac{10}{3},\; \tfrac53\right],
$$

$$
R_3 \leftarrow R_3 - \tfrac23 R_1 = \left[0,\; 1 - \tfrac43,\; 5 - \tfrac23\right]
= \left[0,\; -\tfrac13,\; \tfrac{13}{3}\right] .
$$

$$
\boxed{\text{pivot row } 2, \quad l_{21} = \tfrac13, \quad l_{31} = \tfrac23, \quad
A^{(1)} = \begin{bmatrix} 3 & 2 & 1 \\ 0 & \tfrac{10}{3} & \tfrac53 \\ 0 & -\tfrac13 & \tfrac{13}{3} \end{bmatrix}}
$$

**Key takeaway.** Partial pivoting is a search over one column, $O(n)$ comparisons per step, and
it is what makes $\lvert l_{ij} \rvert \le 1$ in Proof 5.5.

In [13]:
A = np.array([[1.0, 4.0, 2.0], [3.0, 2.0, 1.0], [2.0, 1.0, 5.0]])
p = int(np.argmax(np.abs(A[:, 0])))
order = list(range(3))
order[0], order[p] = order[p], order[0]
P1 = np.eye(3)[order]
PA = P1 @ A
mult = PA[1:, 0] / PA[0, 0]
A1 = PA.copy()
A1[1:, :] -= np.outer(mult, PA[0, :])
print("pivot row (0-based) =", p, "  multipliers =", mult)
print("A^(1) =\n", A1)
assert p == 1
assert np.allclose(mult, [1 / 3, 2 / 3])
assert np.allclose(A1[2], [0.0, -1 / 3, 13 / 3])
assert np.abs(mult).max() <= 1.0

pivot row (0-based) = 1   multipliers = [0.3333 0.6667]
A^(1) =
 [[ 3.      2.      1.    ]
 [ 0.      3.3333  1.6667]
 [ 0.     -0.3333  4.3333]]


### Problem L1.5 — The flop count of LU

**Statement.** Derive the leading term in the number of floating-point operations used by LU
factorization of an $n \times n$ matrix.

**Intuition.** Step $k$ touches a square block of side $n-k$, and the block shrinks by one each
step.

**Solution.**

*Step 1 — cost of step $k$.* Computing the $n-k$ multipliers costs $n-k$ divisions. Updating
the trailing $(n-k) \times (n-k)$ block costs one multiply and one subtract per entry:

$$
(n-k) + 2(n-k)^2 .
$$

*Step 2 — sum.* With $m = n-k$,

$$
\sum_{k=1}^{n-1} \bigl[ m + 2m^2 \bigr]
= \frac{(n-1)n}{2} + 2\,\frac{(n-1)n(2n-1)}{6} .
$$

*Step 3 — leading term.* The second sum dominates and equals
$\tfrac23 n^3 - n^2 + \tfrac13 n$.

$$
\boxed{\text{flops}_{\mathrm{LU}} = \tfrac23 n^3 + O(n^2)}
$$

**Key takeaway.** Doubling $n$ multiplies the work by eight. The two triangular solves that
follow cost only $2n^2$, which is why the factorization is reused rather than repeated.

In [14]:
def lu_flops_exact(n):
    return sum((n - k) + 2 * (n - k) ** 2 for k in range(1, n))


for n in (10, 50, 100, 500):
    exact = lu_flops_exact(n)
    print(f"n = {n:4d}   exact = {exact:12d}   (2/3) n^3 = {2 / 3 * n ** 3:14.1f}"
          f"   ratio = {exact / (2 / 3 * n ** 3):.4f}")
assert abs(lu_flops_exact(500) / (2 / 3 * 500 ** 3) - 1) < 0.01

n =   10   exact =          615   (2/3) n^3 =          666.7   ratio = 0.9225
n =   50   exact =        82075   (2/3) n^3 =        83333.3   ratio = 0.9849
n =  100   exact =       661650   (2/3) n^3 =       666666.7   ratio = 0.9925
n =  500   exact =     83208250   (2/3) n^3 =     83333333.3   ratio = 0.9985


### Problem L1.6 — $LDL^\top$ of a symmetric $2 \times 2$

**Statement.** Compute the $LDL^\top$ factorization of
$A = \left[\begin{smallmatrix}4 & 12 \\ 12 & 45\end{smallmatrix}\right]$.

**Intuition.** Symmetry means the upper factor is the transpose of the lower one, so only the
pivots and the multipliers are unknown.

**Solution.**

*Step 1.* $d_1 = a_{11} = 4$.

*Step 2.* $l_{21}d_1 = a_{21}$, so $l_{21} = 12/4 = 3$.

*Step 3.* $l_{21}^2 d_1 + d_2 = a_{22}$, so $d_2 = 45 - 9 \cdot 4 = 9$.

$$
\boxed{L = \begin{bmatrix} 1 & 0 \\ 3 & 1 \end{bmatrix}, \qquad
D = \begin{bmatrix} 4 & 0 \\ 0 & 9 \end{bmatrix}}
$$

**Key takeaway.** Both pivots are positive, so by Theorem 4.7 $A$ is positive definite, and the
Cholesky factor is $LD^{1/2} = \left[\begin{smallmatrix}2 & 0 \\ 6 & 3\end{smallmatrix}\right]$.

In [15]:
A = np.array([[4.0, 12.0], [12.0, 45.0]])
L = np.array([[1.0, 0.0], [3.0, 1.0]])
D = np.diag([4.0, 9.0])
print("||A - L D L^T||_F =", np.linalg.norm(A - L @ D @ L.T))
print("L D^{1/2} =\n", L @ np.sqrt(D), "\nnumpy cholesky =\n", np.linalg.cholesky(A))
assert np.allclose(A, L @ D @ L.T)
assert np.allclose(L @ np.sqrt(D), np.linalg.cholesky(A))

||A - L D L^T||_F = 0.0
L D^{1/2} =
 [[2. 0.]
 [6. 3.]] 
numpy cholesky =
 [[2. 0.]
 [6. 3.]]


### Problem L1.7 — The determinant from the LU factors

**Statement.** Show that $PA = LU$ with $s$ row interchanges gives
$\det A = (-1)^s \prod_i u_{ii}$, and evaluate it for the matrix of Problem L1.1.

**Intuition.** The determinant of a triangular matrix is the product of its diagonal, and $L$
has diagonal all ones.

**Solution.** Two standard properties of the determinant are used: multiplicativity, and that
the determinant of a triangular matrix is the product of its diagonal entries.

*Step 1.* Take determinants of $PA = LU$:
$\det P \cdot \det A = \det L \cdot \det U$.

*Step 2.* $\det L = 1$, $\det U = \prod_i u_{ii}$, and $\det P = (-1)^s$ because $P$ is a
product of $s$ transpositions.

*Step 3.* Divide by $\det P = (-1)^s = 1/(-1)^s$.

*Step 4.* For $A$ of Problem L1.1 there are no swaps, so $s = 0$ and
$\det A = 2 \cdot 1 \cdot 2 = 4$.

$$
\boxed{\det A = (-1)^s \prod_{i=1}^{n} u_{ii} = 4}
$$

**Key takeaway.** Determinants are computed in $O(n^3)$ from the factorization, never by
cofactor expansion, which costs $O(n!)$. Cramer's rule is worse still: $n+1$ determinants, so
$O(n^4)$ even with the fast determinant.

In [16]:
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])

# unpivoted elimination, matching the hand computation of Problem L1.1
Uh = A.copy()
for k in range(2):
    Uh[k + 1:, k:] -= np.outer(Uh[k + 1:, k] / Uh[k, k], Uh[k, k:])
print("no swaps (s = 0): diag(U) =", np.diag(Uh),
      "  product =", np.prod(np.diag(Uh)))

# LAPACK's pivoted factorization: a different U, the same signed product
P, L, U = sla.lu(A)
det_from_lu = np.linalg.det(P) * np.prod(np.diag(U))
print("with partial pivoting: diag(U) =", np.diag(U), "  det(P) =", np.linalg.det(P))
print("det from LU =", det_from_lu, "   numpy det =", np.linalg.det(A))
assert np.isclose(np.prod(np.diag(Uh)), 4.0)
assert np.isclose(det_from_lu, np.linalg.det(A))
assert np.isclose(np.linalg.det(A), 4.0)

no swaps (s = 0): diag(U) = [2. 1. 2.]   product = 4.0
with partial pivoting: diag(U) = [ 8.     -0.75   -0.6667]   det(P) = 1.0
det from LU = 4.000000000000002    numpy det = 4.000000000000002


### Problem L1.8 — Inverse of a unit lower triangular matrix

**Statement.** Compute $L^{-1}$ for
$L = \left[\begin{smallmatrix}1&0&0\\2&1&0\\-1&3&1\end{smallmatrix}\right]$.

**Intuition.** The inverse is again unit lower triangular, so only three entries are unknown.

**Solution.**

*Step 1.* Write $L^{-1} = \left[\begin{smallmatrix}1&0&0\\a&1&0\\b&c&1\end{smallmatrix}\right]$
and expand $LL^{-1} = I$.

*Step 2.* Entry $(2,1)$: $2 + a = 0$, so $a = -2$.

*Step 3.* Entry $(3,2)$: $3 + c = 0$, so $c = -3$.

*Step 4.* Entry $(3,1)$: $-1 + 3a + b = 0$, so $b = 1 - 3(-2) = 7$.

$$
\boxed{L^{-1} = \begin{bmatrix} 1 & 0 & 0 \\ -2 & 1 & 0 \\ 7 & -3 & 1 \end{bmatrix}}
$$

**Key takeaway.** Triangularity is preserved under inversion, which is why forward substitution
never needs $L^{-1}$ formed explicitly — and forming it would cost $\tfrac13 n^3$ flops for no
gain.

In [17]:
L = np.array([[1.0, 0.0, 0.0], [2.0, 1.0, 0.0], [-1.0, 3.0, 1.0]])
Linv = np.array([[1.0, 0.0, 0.0], [-2.0, 1.0, 0.0], [7.0, -3.0, 1.0]])
print("L L^-1 =\n", L @ Linv)
print("numpy inv =\n", np.linalg.inv(L))
assert np.allclose(L @ Linv, np.eye(3))
assert np.allclose(np.linalg.inv(L), Linv)

L L^-1 =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
numpy inv =
 [[ 1.  0.  0.]
 [-2.  1.  0.]
 [ 7. -3.  1.]]


### Problem L1.9 — A residual bound with the condition number

**Statement.** Let $\hat{x}$ approximate the solution of $Ax = b$ and $r = b - A\hat{x}$. Prove

$$
\frac{\lVert x - \hat{x} \rVert}{\lVert x \rVert} \le \kappa(A)\,\frac{\lVert r \rVert}{\lVert b \rVert} .
$$

**Intuition.** The residual is the error seen through $A$; undoing $A$ costs a factor
$\lVert A^{-1} \rVert$.

**Solution.**

*Step 1.* Subtracting $A\hat{x}$ from $Ax = b$ gives $A(x - \hat{x}) = r$, so
$x - \hat{x} = A^{-1}r$ and

$$
\lVert x - \hat{x} \rVert \le \lVert A^{-1} \rVert \, \lVert r \rVert .
$$

*Step 2.* From $b = Ax$, $\lVert b \rVert \le \lVert A \rVert \lVert x \rVert$, hence
$1/\lVert x \rVert \le \lVert A \rVert / \lVert b \rVert$.

*Step 3.* Multiply the two inequalities.

$$
\boxed{\frac{\lVert x - \hat{x} \rVert}{\lVert x \rVert} \le \kappa(A)\,\frac{\lVert r \rVert}{\lVert b \rVert}}
$$

**Key takeaway.** A small residual certifies accuracy only when $\kappa(A)$ is small. Combined
with Theorem 4.11 this is the whole story: the residual measures backward error, and $\kappa$
converts backward error into forward error.

In [18]:
A = np.array([[1.0, 2.0], [1.0, 2.01]])
b = np.array([3.0, 3.01])
x = np.linalg.solve(A, b)
kap = np.linalg.cond(A, np.inf)
for xhat in (np.array([0.0, 1.5]), np.array([1.2, 0.9]), x + 1e-8):
    r = b - A @ xhat
    lhs = np.linalg.norm(x - xhat, np.inf) / np.linalg.norm(x, np.inf)
    rhs = kap * np.linalg.norm(r, np.inf) / np.linalg.norm(b, np.inf)
    print(f"xhat = {xhat}   forward error = {lhs:.6e}   bound = {rhs:.6e}   holds = {lhs <= rhs}")
    assert lhs <= rhs * (1 + 1e-12)

xhat = [0.  1.5]   forward error = 1.000000e+00   bound = 2.005000e+00   holds = True
xhat = [1.2 0.9]   forward error = 2.000000e-01   bound = 4.010000e-01   holds = True
xhat = [1. 1.]   forward error = 1.000000e-08   bound = 1.207010e-05   holds = True


### Problem L1.10 — Cholesky of a $3 \times 3$ SPD matrix

**Statement.** Compute the Cholesky factor of

$$
A = \begin{bmatrix} 4 & 2 & 6 \\ 2 & 10 & 9 \\ 6 & 9 & 29 \end{bmatrix} .
$$

**Intuition.** Column $k$ of $L$ is determined by column $k$ of $A$ minus what earlier columns
of $L$ already account for.

**Solution.**

*Step 1 — column 1.* $l_{11} = \sqrt{4} = 2$, $l_{21} = 2/2 = 1$, $l_{31} = 6/2 = 3$.

*Step 2 — column 2.* $l_{22} = \sqrt{10 - 1^2} = 3$ and $l_{32} = (9 - 3 \cdot 1)/3 = 2$.

*Step 3 — column 3.* $l_{33} = \sqrt{29 - 3^2 - 2^2} = 4$.

$$
\boxed{L = \begin{bmatrix} 2 & 0 & 0 \\ 1 & 3 & 0 \\ 3 & 2 & 4 \end{bmatrix}}
$$

**Key takeaway.** All three radicands were positive, which by Theorem 4.6 proves $A$ is
positive definite without computing a single eigenvalue.

In [19]:
A = np.array([[4.0, 2.0, 6.0], [2.0, 10.0, 9.0], [6.0, 9.0, 29.0]])
L = np.array([[2.0, 0.0, 0.0], [1.0, 3.0, 0.0], [3.0, 2.0, 4.0]])
print("||A - L L^T||_F =", np.linalg.norm(A - L @ L.T))
print("radicands:", [4.0, 10 - 1, 29 - 9 - 4], "  eigenvalues:", np.linalg.eigvalsh(A))
assert np.allclose(np.linalg.cholesky(A), L)
assert (np.linalg.eigvalsh(A) > 0).all()

||A - L L^T||_F = 0.0
radicands: [4.0, 9, 16]   eigenvalues: [ 2.6341  6.4469 33.919 ]


### Problem L1.11 — The flop count of Cholesky

**Statement.** Show that Cholesky factorization of an $n \times n$ SPD matrix costs about
$\tfrac13 n^3$ flops, half of LU.

**Intuition.** Symmetry means only the lower triangle is computed.

**Solution.**

*Step 1 — cost of column $k$.* The diagonal entry
$l_{kk} = \sqrt{a_{kk} - \sum_{j \lt k} l_{kj}^2}$ costs $2(k-1)$ flops plus one square root.
Each of the $n-k$ subdiagonal entries

$$
l_{ik} = \frac{1}{l_{kk}}\left( a_{ik} - \sum_{j \lt k} l_{ij}l_{kj} \right)
$$

costs $2(k-1)$ flops plus one division.

*Step 2 — sum.* Ignoring lower-order terms,

$$
\sum_{k=1}^{n} 2(n-k)(k-1) \approx 2\sum_{k=1}^{n}(nk - k^2)
= 2\left( \frac{n^3}{2} - \frac{n^3}{3} \right) = \frac{n^3}{3} .
$$

$$
\boxed{\text{flops}_{\mathrm{Cholesky}} \approx \tfrac13 n^3 = \tfrac12 \,\text{flops}_{\mathrm{LU}}}
$$

**Key takeaway.** Symmetry buys a factor of two in time and in storage, and positive
definiteness additionally removes the need to pivot for stability.

In [20]:
def chol_flops_exact(n):
    return sum(2 * (k - 1) + (n - k) * 2 * (k - 1) for k in range(1, n + 1))


for n in (10, 50, 100, 500):
    exact = chol_flops_exact(n)
    print(f"n = {n:4d}   exact = {exact:12d}   n^3/3 = {n ** 3 / 3:14.1f}"
          f"   ratio = {exact / (n ** 3 / 3):.4f}")
print("ratio LU / Cholesky at n = 500:",
      lu_flops_exact(500) / chol_flops_exact(500))
assert abs(chol_flops_exact(500) / (500 ** 3 / 3) - 1) < 0.01
assert 1.9 < lu_flops_exact(500) / chol_flops_exact(500) < 2.1

n =   10   exact =          330   n^3/3 =          333.3   ratio = 0.9900
n =   50   exact =        41650   n^3/3 =        41666.7   ratio = 0.9996
n =  100   exact =       333300   n^3/3 =       333333.3   ratio = 0.9999
n =  500   exact =     41666500   n^3/3 =     41666666.7   ratio = 1.0000
ratio LU / Cholesky at n = 500: 1.9970059880239521


### Problem L1.12 — Dimensions of the four fundamental subspaces

**Statement.** Let $A \in \mathbb{R}^{4 \times 5}$ have rank $r = 3$. Give the dimensions of
$\operatorname{Col}(A)$, $\operatorname{Col}(A^\top)$, $\operatorname{Null}(A)$ and
$\operatorname{Null}(A^\top)$.

**Intuition.** Rank counts independent directions; everything else is orthogonal complement.

**Solution.**

*Step 1.* $\dim \operatorname{Col}(A) = r = 3$, a subspace of $\mathbb{R}^4$.

*Step 2.* $\dim \operatorname{Col}(A^\top) = r = 3$ by Proposition 4.1, a subspace of
$\mathbb{R}^5$.

*Step 3.* $\dim \operatorname{Null}(A) = n - r = 5 - 3 = 2$.

*Step 4.* $\dim \operatorname{Null}(A^\top) = m - r = 4 - 3 = 1$.

$$
\boxed{3,\; 3,\; 2,\; 1 \text{ in } \mathbb{R}^4,\, \mathbb{R}^5,\, \mathbb{R}^5,\, \mathbb{R}^4}
$$

**Key takeaway.** The two sums $3 + 2 = 5$ and $3 + 1 = 4$ are the orthogonal decompositions of
Theorem 4.3, and both rest on Proposition 4.1.

In [21]:
B = rng.standard_normal((4, 3))
C = rng.standard_normal((3, 5))
A = B @ C
r = np.linalg.matrix_rank(A)
dims = (int(np.linalg.matrix_rank(A)), int(np.linalg.matrix_rank(A.T)),
        int(5 - r), int(4 - r))
print("rank(A) =", r, "  dims (Col, Row, Null, LeftNull) =", dims)
ns = sla.null_space(A)
lns = sla.null_space(A.T)
print("null_space shapes:", ns.shape, lns.shape)
print("max |A @ n| over null basis:", np.abs(A @ ns).max())
assert dims == (3, 3, 2, 1)
assert ns.shape[1] == 2 and lns.shape[1] == 1

rank(A) = 3   dims (Col, Row, Null, LeftNull) = (3, 3, 2, 1)
null_space shapes: (5, 2) (4, 1)
max |A @ n| over null basis: 4.57812078099657e-16


### Problem L1.13 — Solvability classified by rank

**Statement.** For $A \in \mathbb{R}^{m \times n}$ of rank $r$, classify the number of
solutions of $Ax = b$ in the four cases (i) $r = m = n$, (ii) $r = m \lt n$, (iii)
$r = n \lt m$, (iv) $r \lt m$ and $r \lt n$.

**Intuition.** Full row rank gives existence; full column rank gives uniqueness.

**Solution.**

*Step 1 — case (i).* $\operatorname{Col}(A) = \mathbb{R}^m$ and
$\operatorname{Null}(A) = \lbrace 0 \rbrace$: exactly one solution for every $b$.

*Step 2 — case (ii).* $\operatorname{Col}(A) = \mathbb{R}^m$ but
$\dim \operatorname{Null}(A) = n - m \gt 0$: infinitely many solutions for every $b$.

*Step 3 — case (iii).* $\operatorname{Col}(A) \subsetneq \mathbb{R}^m$ and
$\operatorname{Null}(A) = \lbrace 0 \rbrace$: no solution unless
$b \in \operatorname{Col}(A)$, and then exactly one.

*Step 4 — case (iv).* Neither: no solution unless $b \in \operatorname{Col}(A)$, and then
infinitely many.

$$
\boxed{\text{(i) } 1, \quad \text{(ii) } \infty, \quad \text{(iii) } 0 \text{ or } 1, \quad \text{(iv) } 0 \text{ or } \infty}
$$

**Key takeaway.** Existence is a statement about rows, uniqueness a statement about columns, and
Theorem 4.2 is the bookkeeping that joins them.

In [22]:
cases = {
    "(i)  r = m = n": np.array([[1.0, 0.0], [0.0, 1.0]]),
    "(ii) r = m < n": np.array([[1.0, 0.0, 2.0], [0.0, 1.0, 3.0]]),
    "(iii) r = n < m": np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]),
    "(iv) r < m, r < n": np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]]),
}
for name, M in cases.items():
    m, n = M.shape
    r = np.linalg.matrix_rank(M)
    b_in = M @ np.ones(n)
    b_out = b_in + sla.null_space(M.T)[:, 0] if r < m else None
    solv_in = np.linalg.matrix_rank(np.column_stack([M, b_in])) == r
    solv_out = (np.linalg.matrix_rank(np.column_stack([M, b_out])) == r
                if b_out is not None else None)
    off = "no b lies off Col(A)" if solv_out is None else str(solv_out)
    print(f"{name:20s} m={m} n={n} r={r}  dim Null = {n - r}"
          f"   b in Col(A) solvable: {solv_in}   b off Col(A) solvable: {off}")
    assert solv_in
    if solv_out is not None:
        assert not solv_out

(i)  r = m = n       m=2 n=2 r=2  dim Null = 0   b in Col(A) solvable: True   b off Col(A) solvable: no b lies off Col(A)
(ii) r = m < n       m=2 n=3 r=2  dim Null = 1   b in Col(A) solvable: True   b off Col(A) solvable: no b lies off Col(A)
(iii) r = n < m      m=3 n=2 r=2  dim Null = 0   b in Col(A) solvable: True   b off Col(A) solvable: False
(iv) r < m, r < n    m=3 n=2 r=1  dim Null = 1   b in Col(A) solvable: True   b off Col(A) solvable: False


### Problem L1.14 — The complete solution set

**Statement.** Find every solution of

$$
\begin{bmatrix} 1 & 2 & 1 \\ 2 & 4 & 5 \end{bmatrix} x = \begin{bmatrix} 3 \\ 9 \end{bmatrix},
$$

in the form $x = x_p + c\,v_n$. (This is the system of Example 6.1; here the reduction is
carried all the way to reduced row echelon form.)

**Intuition.** A particular solution plus the null space is the whole solution set.

**Solution.**

*Step 1 — eliminate.* $R_2 \leftarrow R_2 - 2R_1$ gives $[0, 0, 3 \mid 3]$.

*Step 2 — normalize and clear upward.* Scaling row $2$ by $\tfrac13$ and subtracting it from
row $1$:

$$
\begin{bmatrix} 1 & 2 & 0 & \mid & 2 \\ 0 & 0 & 1 & \mid & 1 \end{bmatrix} .
$$

*Step 3 — free variable.* Columns $1$ and $3$ hold pivots, so $x_2 = c$ is free, $x_3 = 1$ and
$x_1 = 2 - 2c$.

$$
\boxed{x = \begin{bmatrix} 2 \\ 0 \\ 1 \end{bmatrix} + c \begin{bmatrix} -2 \\ 1 \\ 0 \end{bmatrix}, \qquad c \in \mathbb{R}}
$$

**Key takeaway.** The solution set is an affine line: a point plus a one-dimensional null space,
of dimension $n - r = 3 - 2 = 1$ exactly as Theorem 4.2 predicts.

In [23]:
A = np.array([[1.0, 2.0, 1.0], [2.0, 4.0, 5.0]])
b = np.array([3.0, 9.0])
xp = np.array([2.0, 0.0, 1.0])
vn = np.array([-2.0, 1.0, 0.0])
ns = sla.null_space(A)
print("rank(A) =", np.linalg.matrix_rank(A), "  dim Null(A) =", ns.shape[1])
print("A xp - b =", A @ xp - b, "   A vn =", A @ vn)
print("vn parallel to the computed null basis:",
      np.allclose(np.abs(ns[:, 0] @ (vn / np.linalg.norm(vn))), 1.0))
for c in rng.standard_normal(4):
    assert np.allclose(A @ (xp + c * vn), b)
assert ns.shape[1] == 1

rank(A) = 2   dim Null(A) = 1
A xp - b = [0. 0.]    A vn = [0. 0.]
vn parallel to the computed null basis: True


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Steady heat conduction on a rod (physics)

**Statement.** A thin insulated rod occupies $[0,1]$ with both ends held at temperature $0$.
The steady temperature $u$ satisfies $-u'' = f$. Discretizing with three interior nodes at
spacing $h = \tfrac14$ and writing $b_i = h^2 f(x_i)$ turns this into

$$
\begin{bmatrix} 2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2 \end{bmatrix}
\begin{bmatrix} u_1 \\ u_2 \\ u_3 \end{bmatrix}
= \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix} .
$$

Solve it with the Thomas algorithm — LU factorization specialized to a tridiagonal matrix — and
state its cost.

**Intuition.** Each node couples only to its neighbours, so elimination never creates an entry
outside the band and the whole factorization is linear in $n$.

**Solution.** Write $d = (2,2,2)$ for the diagonal, $a = (-1,-1)$ for the subdiagonal and
$c = (-1,-1)$ for the superdiagonal.

*Step 1 — forward elimination.*

$$
d_1' = 2, \qquad b_1' = 1 .
$$

$$
m_2 = \frac{a_1}{d_1'} = -\tfrac12, \quad
d_2' = 2 - \left(-\tfrac12\right)(-1) = \tfrac32, \quad
b_2' = 0 - \left(-\tfrac12\right)(1) = \tfrac12 .
$$

$$
m_3 = \frac{a_2}{d_2'} = -\tfrac23, \quad
d_3' = 2 - \left(-\tfrac23\right)(-1) = \tfrac43, \quad
b_3' = 1 - \left(-\tfrac23\right)\left(\tfrac12\right) = \tfrac43 .
$$

*Step 2 — back substitution.*

$$
u_3 = \frac{4/3}{4/3} = 1, \qquad
u_2 = \frac{\tfrac12 + 1}{\tfrac32} = 1, \qquad
u_1 = \frac{1 + 1}{2} = 1 .
$$

*Step 3 — cost.* Each of the $n-1$ elimination steps touches three numbers, and each of the
$n$ back-substitution steps touches two, so the total is $O(n)$ flops and $O(n)$ storage —
against $\tfrac23 n^3$ and $n^2$ for a dense solve.

$$
\boxed{u = (1,\; 1,\; 1)^\top, \qquad \text{cost } O(n) \text{ flops}}
$$

**Key takeaway.** The matrix is symmetric positive definite and tridiagonal, so by the
bandwidth argument of Problem L3.1 the Cholesky factor is bidiagonal, no pivoting is needed for
stability, and the solver is linear in the number of nodes.

In [24]:
def thomas(d, a, c, b):
    """Tridiagonal solve: subdiagonal a, diagonal d, superdiagonal c."""
    n = len(d)
    dp, bp = d.astype(float).copy(), b.astype(float).copy()
    for k in range(1, n):
        m = a[k - 1] / dp[k - 1]
        dp[k] -= m * c[k - 1]
        bp[k] -= m * bp[k - 1]
    x = np.empty(n)
    x[-1] = bp[-1] / dp[-1]
    for k in range(n - 2, -1, -1):
        x[k] = (bp[k] - c[k] * x[k + 1]) / dp[k]
    return x


d = np.array([2.0, 2.0, 2.0])
a = np.array([-1.0, -1.0])
c = np.array([-1.0, -1.0])
b = np.array([1.0, 0.0, 1.0])
u = thomas(d, a, c, b)
A3 = np.diag(d) + np.diag(a, -1) + np.diag(c, 1)
print("Thomas solution   =", u)
print("dense solve       =", np.linalg.solve(A3, b))
print("residual          =", np.linalg.norm(A3 @ u - b))
assert np.allclose(u, [1.0, 1.0, 1.0])

m = 400
Abig = (np.diag(2.0 * np.ones(m)) - np.diag(np.ones(m - 1), 1) - np.diag(np.ones(m - 1), -1))
bbig = np.ones(m)
u_th = thomas(2.0 * np.ones(m), -np.ones(m - 1), -np.ones(m - 1), bbig)
u_dn = np.linalg.solve(Abig, bbig)
Lbig = np.linalg.cholesky(Abig)
bw = int(max(abs(i - j) for i in range(m) for j in range(m)
             if abs(Lbig[i, j]) > 1e-10 * np.abs(Lbig).max()))
print(f"m = {m}: ||thomas - dense|| = {np.linalg.norm(u_th - u_dn):.3e}"
      f"   Cholesky factor bandwidth = {bw}")
assert np.allclose(u_th, u_dn)
assert bw == 1

Thomas solution   = [1. 1. 1.]
dense solve       = [1. 1. 1.]
residual          = 2.482534153247273e-16


m = 400: ||thomas - dense|| = 0.000e+00   Cholesky factor bandwidth = 1


### Problem L2.2 — A resistor network as a Laplacian system (physics)

**Statement.** Four nodes are joined by unit-conductance resistors along the edges
$\lbrace 1,2 \rbrace$, $\lbrace 2,3 \rbrace$, $\lbrace 3,4 \rbrace$, $\lbrace 4,1 \rbrace$ and
$\lbrace 1,3 \rbrace$. A current of $1$ ampere enters at node $1$ and leaves at node $4$.
Kirchhoff's laws give $Lv = b$ with $L = D - W$ the graph Laplacian and
$b = (1,0,0,-1)^\top$.

Show $\operatorname{Null}(L) = \operatorname{span}(\mathbf{1})$, solve for the potentials with
node $4$ grounded, and give the effective resistance between nodes $1$ and $4$.

**Intuition.** Potentials are determined only up to a constant offset, which is exactly the
one-dimensional null space; grounding a node removes it.

**Solution.**

*Step 1 — the null space.* For any $v$,

$$
v^\top L v = \sum_{\lbrace i,j \rbrace \in E} (v_i - v_j)^2 \ \ge 0 ,
$$

and it vanishes only when $v_i = v_j$ along every edge. The graph is connected, so
$v = c\mathbf{1}$, and $L\mathbf{1} = 0$ confirms the reverse inclusion. Hence
$\operatorname{rank} L = 3$.

*Step 2 — solvability.* $\operatorname{Col}(L) = \operatorname{Null}(L^\top)^{\perp} = \mathbf{1}^{\perp}$
by Theorem 4.3, and $\mathbf{1}^\top b = 0$: injected current balances, so a solution exists.

*Step 3 — ground a node.* Set $v_4 = 0$ and delete row and column $4$. The reduced
$\tilde{L} \in \mathbb{R}^{3 \times 3}$ is symmetric with strictly positive pivots, so Cholesky
applies:

$$
\tilde{L} = \begin{bmatrix} 3 & -1 & -1 \\ -1 & 2 & -1 \\ -1 & -1 & 3 \end{bmatrix},
\qquad
\tilde{v} = \left(\tfrac58,\; \tfrac12,\; \tfrac38\right)^\top .
$$

*Step 4 — effective resistance.* With $1$ ampere injected,
$R_{\mathrm{eff}} = v_1 - v_4 = \tfrac58$.

$$
\boxed{v = \left(\tfrac58,\; \tfrac12,\; \tfrac38,\; 0\right)^\top, \qquad R_{14} = \tfrac58 = 0.625\ \Omega}
$$

**Key takeaway.** Grounding one node is the physical name for removing a null vector; the
resulting matrix is positive definite, so the electrical problem is solved by exactly the
Cholesky of Theorem 4.6.

In [25]:
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)]
W = np.zeros((4, 4))
for i, j in edges:
    W[i, j] = W[j, i] = 1.0
Lap = np.diag(W.sum(axis=1)) - W
b = np.array([1.0, 0.0, 0.0, -1.0])
print("Laplacian =\n", Lap)
print("L 1 =", Lap @ np.ones(4), "   eigenvalues =", np.linalg.eigvalsh(Lap))
print("1^T b =", np.ones(4) @ b, " -> b is in Col(L)")
Lr = Lap[:3, :3]
Lchol = np.linalg.cholesky(Lr)
v = np.zeros(4)
v[:3] = sla.cho_solve((Lchol, True), b[:3])
print("potentials v =", v)
print("residual ||L v - b|| =", np.linalg.norm(Lap @ v - b))
print("effective resistance R_14 =", v[0] - v[3], " = 5/8 =", 5 / 8)
assert np.allclose(v, [5 / 8, 1 / 2, 3 / 8, 0.0])
assert np.isclose(v[0] - v[3], 0.625)
assert np.linalg.matrix_rank(Lap) == 3

Laplacian =
 [[ 3. -1. -1. -1.]
 [-1.  2. -1.  0.]
 [-1. -1.  3. -1.]
 [-1.  0. -1.  2.]]
L 1 = [0. 0. 0. 0.]    eigenvalues = [0. 2. 4. 4.]
1^T b = 0.0  -> b is in Col(L)
potentials v = [0.625 0.5   0.375 0.   ]
residual ||L v - b|| = 5.978733960281817e-16
effective resistance R_14 = 0.6250000000000002  = 5/8 = 0.625


### Problem L2.3 — The condition number of a near-singular calibration matrix

**Statement.** Two sensors respond to the same two inputs with almost identical gains:

$$
A = \begin{bmatrix} 1 & 2 \\ 1 & 2.01 \end{bmatrix} .
$$

Compute $\kappa_\infty(A) = \lVert A \rVert_\infty \lVert A^{-1} \rVert_\infty$.

**Intuition.** Nearly identical rows make $\det A$ tiny relative to the entries, and the inverse
huge.

**Solution.**

*Step 1 — $\lVert A \rVert_\infty$.* Row sums are $3$ and $3.01$, so
$\lVert A \rVert_\infty = 3.01$.

*Step 2 — the inverse.* $\det A = 2.01 - 2 = 0.01$, so

$$
A^{-1} = \frac{1}{0.01}\begin{bmatrix} 2.01 & -2 \\ -1 & 1 \end{bmatrix}
= \begin{bmatrix} 201 & -200 \\ -100 & 100 \end{bmatrix} .
$$

*Step 3 — $\lVert A^{-1} \rVert_\infty$.* Row sums are $401$ and $200$, so the norm is $401$.

$$
\boxed{\kappa_\infty(A) = 3.01 \times 401 = 1207.01}
$$

**Key takeaway.** $\kappa \approx 1.2 \times 10^3$ means about three decimal digits are at risk.
With sensors calibrated to four digits, the recovered inputs are trustworthy to roughly one.

In [26]:
A = np.array([[1.0, 2.0], [1.0, 2.01]])
Ainv = np.linalg.inv(A)
kap = np.linalg.norm(A, np.inf) * np.linalg.norm(Ainv, np.inf)
print("||A||_inf =", np.linalg.norm(A, np.inf), "  A^-1 =\n", Ainv)
print("||A^-1||_inf =", np.linalg.norm(Ainv, np.inf), "  kappa_inf =", kap)
print("numpy cond(A, inf) =", np.linalg.cond(A, np.inf),
      "  kappa_2 =", np.linalg.cond(A))
assert np.isclose(kap, 1207.01)
assert np.isclose(np.linalg.cond(A, np.inf), 1207.01)

||A||_inf = 3.01   A^-1 =
 [[ 201. -200.]
 [-100.  100.]]
||A^-1||_inf = 401.0000000000085   kappa_inf = 1207.0100000000257
numpy cond(A, inf) = 1207.0100000000257   kappa_2 = 1004.0090039930228


### Problem L2.4 — A $0.33$ percent data error becomes a $200$ percent answer error

**Statement.** With $A$ as in Problem L2.3 and $b = (3, 3.01)^\top$ the exact solution is
$x = (1,1)^\top$. Replace $b$ by $\hat{b} = (3, 3.02)^\top$ and compare the relative change in
the data with the relative change in the answer.

**Intuition.** The perturbation moves the second line by a hair, and the near-parallel
intersection slides a long way.

**Solution.**

*Step 1.* $\delta b = (0, 0.01)^\top$ and

$$
\delta x = A^{-1}\delta b = \begin{bmatrix} 201 & -200 \\ -100 & 100 \end{bmatrix}
\begin{bmatrix} 0 \\ 0.01 \end{bmatrix} = \begin{bmatrix} -2 \\ 1 \end{bmatrix} .
$$

*Step 2.* Hence $\hat{x} = (-1, 2)^\top$.

*Step 3.* Relative changes in the $\infty$-norm:

$$
\frac{\lVert \delta b \rVert_\infty}{\lVert b \rVert_\infty} = \frac{0.01}{3.01} \approx 0.33\%,
\qquad
\frac{\lVert \delta x \rVert_\infty}{\lVert x \rVert_\infty} = 2 = 200\% .
$$

*Step 4.* The amplification is $2 \times 301 = 602$, under the ceiling
$\kappa_\infty(A) = 1207.01$ of Theorem 4.10.

$$
\boxed{\hat{x} = (-1,\; 2)^\top, \qquad \text{amplification } 602 \le \kappa_\infty = 1207.01}
$$

**Key takeaway.** Nothing went wrong in the arithmetic: the exact solution of the perturbed
data really is $(-1,2)$. Ill-conditioning is a property of the question, not of the answer.

In [27]:
b = np.array([3.0, 3.01])
x = np.array([1.0, 1.0])
db = np.array([0.0, 0.01])
dx = Ainv @ db
xhat = x + dx
rel_b = np.linalg.norm(db, np.inf) / np.linalg.norm(b, np.inf)
rel_x = np.linalg.norm(dx, np.inf) / np.linalg.norm(x, np.inf)
print("xhat =", xhat, "   exact solve =", np.linalg.solve(A, b + db))
print(f"rel change in b = {rel_b:.6f}   rel change in x = {rel_x:.6f}")
print(f"amplification = {rel_x / rel_b:.2f}   kappa_inf = {kap:.2f}")
assert np.allclose(xhat, [-1.0, 2.0])
assert np.isclose(rel_x / rel_b, 602.0)
assert rel_x / rel_b <= kap

xhat = [-1.  2.]    exact solve = [-1.  2.]
rel change in b = 0.003322   rel change in x = 2.000000
amplification = 602.00   kappa_inf = 1207.01


### Problem L2.5 — LU or QR: cost against stability

**Statement.** Compare LU with partial pivoting against Householder QR for solving a square
system $Ax = b$: flop count and stability. State precisely what "stable" means for each.

**Intuition.** QR replaces the possibly growing multipliers of elimination by orthogonal
reflections, which cannot change any norm.

**Solution.**

*Step 1 — cost.* LU with partial pivoting costs $\tfrac23 n^3$ flops; Householder QR costs
$\tfrac43 n^3$ for the factorization, plus $O(n^2)$ to form $Q^\top b$ and back-substitute. QR
is about twice the work.

*Step 2 — what LU guarantees.* By Theorem 4.12, the computed $\hat{x}$ satisfies
$(A + \delta A)\hat{x} = b$ with

$$
\frac{\lVert \delta A \rVert_\infty}{\lVert A \rVert_\infty} \le c\,n^3 \rho\, u ,
$$

so LU is backward stable **provided the growth factor $\rho$ stays modest**. It can reach
$2^{n-1}$ in the worst case, though the median for random matrices is under $2$ up to
$n = 14$, as measured in Section 7.5.

*Step 3 — what QR guarantees.* Householder QR gives $\hat{Q}\hat{R} = A + \delta A$ with
$\lVert \delta A \rVert / \lVert A \rVert = O(n\,u)$ **unconditionally** — there is no growth
factor, because $\lVert Q \rVert_2 = 1$ and $\kappa_2(Q) = 1$, so no intermediate quantity can
grow.

*Step 4 — what neither guarantees.* Backward stability is not accuracy. For both methods the
forward error is still bounded only by

$$
\frac{\lVert \hat{x} - x \rVert}{\lVert x \rVert} = O\bigl(\kappa(A)\,u\bigr) ,
$$

by Theorem 4.10. QR is not "unconditionally stable" in the sense of returning an accurate
answer; it is unconditionally *backward* stable.

$$
\boxed{\text{LU } \tfrac23 n^3 \text{ flops, backward stable up to } \rho; \quad
\text{QR } \tfrac43 n^3, \text{ backward stable with no } \rho; \quad
\text{both: forward error } O(\kappa u)}
$$

**Key takeaway.** LU is the default for square systems because $\rho$ is tiny in practice; QR
is the default for least squares, where the relevant condition number is $\kappa_2(A)$ rather
than $\kappa_2(A^\top A) = \kappa_2(A)^2$.

In [28]:
kappas = [1e2, 1e6, 1e10]
print("kappa_2      LU fwd err   QR fwd err   LU bwd err   QR bwd err")
for kap_t in kappas:
    Q1, _ = np.linalg.qr(rng.standard_normal((40, 40)))
    Q2, _ = np.linalg.qr(rng.standard_normal((40, 40)))
    M = Q1 @ np.diag(np.logspace(0, -np.log10(kap_t), 40)) @ Q2.T
    x_true = rng.standard_normal(40)
    rhs = M @ x_true
    x_lu = sla.lu_solve(sla.lu_factor(M), rhs)
    Qh, Rh = np.linalg.qr(M)
    x_qr = sla.solve_triangular(Rh, Qh.T @ rhs, lower=False)
    fe = lambda z: np.linalg.norm(z - x_true) / np.linalg.norm(x_true)
    be = lambda z: (np.linalg.norm(rhs - M @ z)
                    / (np.linalg.norm(M, 2) * np.linalg.norm(z)))
    print(f"{kap_t:9.1e}   {fe(x_lu):.3e}    {fe(x_qr):.3e}    "
          f"{be(x_lu):.3e}    {be(x_qr):.3e}")
    assert be(x_lu) < 1e-13 and be(x_qr) < 1e-13
    assert fe(x_lu) < 50 * kap_t * U_ROUND and fe(x_qr) < 50 * kap_t * U_ROUND

kappa_2      LU fwd err   QR fwd err   LU bwd err   QR bwd err
  1.0e+02   2.445e-15    3.248e-15    1.466e-16    1.270e-16
  1.0e+06   1.491e-11    1.380e-11    6.412e-17    5.855e-17
  1.0e+10   9.191e-08    8.317e-08    4.527e-17    4.153e-17


### Problem L2.6 — Block inversion and Gaussian conditioning

**Statement.** For $M = \left[\begin{smallmatrix}A & B \\ C & D\end{smallmatrix}\right]$ with
$A$ nonsingular and $S = D - CA^{-1}B$ nonsingular, derive $M^{-1}$ blockwise, and identify the
statistical meaning of the blocks when $M = \Sigma$ is a covariance matrix.

**Intuition.** Invert the three factors of Theorem 4.8 in reverse order.

**Solution.**

*Step 1 — invert the factorization.* From Theorem 4.8,
$M = L_{\mathrm{blk}}\operatorname{diag}(A, S)\,U_{\mathrm{blk}}$, so

$$
M^{-1} = U_{\mathrm{blk}}^{-1}\begin{bmatrix} A^{-1} & 0 \\ 0 & S^{-1} \end{bmatrix}L_{\mathrm{blk}}^{-1},
$$

with $L_{\mathrm{blk}}^{-1} = \left[\begin{smallmatrix}I & 0 \\ -CA^{-1} & I\end{smallmatrix}\right]$
and $U_{\mathrm{blk}}^{-1} = \left[\begin{smallmatrix}I & -A^{-1}B \\ 0 & I\end{smallmatrix}\right]$.

*Step 2 — multiply.*

$$
M^{-1} = \begin{bmatrix}
A^{-1} + A^{-1}BS^{-1}CA^{-1} & -A^{-1}BS^{-1} \\
-S^{-1}CA^{-1} & S^{-1}
\end{bmatrix} .
$$

*Step 3 — read the statistics.* Take $M = \Sigma$ the covariance of $(z_1, z_2)$ with
$A = \Sigma_{11}$, $D = \Sigma_{22}$. The bottom-right block of $\Sigma^{-1}$, the precision
matrix, is $S^{-1}$ where

$$
S = \Sigma_{22} - \Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}
$$

is exactly the conditional covariance $\operatorname{Cov}(z_2 \mid z_1)$.

$$
\boxed{M^{-1} = \begin{bmatrix} A^{-1} + A^{-1}BS^{-1}CA^{-1} & -A^{-1}BS^{-1} \\ -S^{-1}CA^{-1} & S^{-1} \end{bmatrix}}
$$

**Key takeaway.** A block of the precision matrix is the inverse of a conditional covariance.
That is why a zero in the precision matrix means conditional independence, and why graphical
models are built on $\Sigma^{-1}$ rather than $\Sigma$.

In [29]:
Sig = np.array([[2.0, 0.5, 0.3, 0.1],
                [0.5, 1.0, 0.2, 0.0],
                [0.3, 0.2, 1.5, 0.4],
                [0.1, 0.0, 0.4, 1.2]])
A_, B_ = Sig[:2, :2], Sig[:2, 2:]
C_, D_ = Sig[2:, :2], Sig[2:, 2:]
S_ = D_ - C_ @ np.linalg.inv(A_) @ B_
Ainv_, Sinv_ = np.linalg.inv(A_), np.linalg.inv(S_)
Minv = np.block([[Ainv_ + Ainv_ @ B_ @ Sinv_ @ C_ @ Ainv_, -Ainv_ @ B_ @ Sinv_],
                 [-Sinv_ @ C_ @ Ainv_, Sinv_]])
print("||block formula - numpy inverse||_F =", np.linalg.norm(Minv - np.linalg.inv(Sig)))
print("conditional covariance Cov(z2|z1) = S =\n", S_)
print("bottom-right block of Sigma^-1    =\n", np.linalg.inv(Sig)[2:, 2:])
print("S is positive definite:", (np.linalg.eigvalsh(S_) > 0).all())
assert np.allclose(Minv, np.linalg.inv(Sig))
assert np.allclose(np.linalg.inv(Sig)[2:, 2:], Sinv_)
assert (np.linalg.eigvalsh(S_) > 0).all()

||block formula - numpy inverse||_F = 1.3243111249278346e-16
conditional covariance Cov(z2|z1) = S =
 [[1.4371 0.3886]
 [0.3886 1.1943]]
bottom-right block of Sigma^-1    =
 [[ 0.7629 -0.2482]
 [-0.2482  0.9181]]
S is positive definite: True


### Problem L2.7 — Sherman-Morrison and recursive least squares

**Statement.** Prove the Sherman-Morrison formula

$$
(A + uv^\top)^{-1} = A^{-1} - \frac{A^{-1}uv^\top A^{-1}}{1 + v^\top A^{-1}u},
\qquad 1 + v^\top A^{-1}u \neq 0,
$$

and use it to update a least-squares solution when one new data row arrives.

**Intuition.** A rank-one change to a matrix produces a rank-one change to its inverse.

**Solution.**

*Step 1 — verify by multiplication.* Write $\beta = 1 + v^\top A^{-1}u$ and multiply out:

$$
(A + uv^\top)\left( A^{-1} - \frac{A^{-1}uv^\top A^{-1}}{\beta} \right)
= I - \frac{uv^\top A^{-1}}{\beta} + uv^\top A^{-1} - \frac{u(v^\top A^{-1}u)v^\top A^{-1}}{\beta} .
$$

*Step 2 — collect.* The scalar $v^\top A^{-1}u = \beta - 1$, so the first and last correction
terms combine to $-\beta \, uv^\top A^{-1}/\beta = -uv^\top A^{-1}$, which cancels the middle
term and leaves $I$.

*Step 3 — apply it.* In recursive least squares the normal-equation matrix after $k$ rows is
$G_k = X_k^\top X_k$. A new row $a^\top$ gives $G_{k+1} = G_k + aa^\top$, so with
$P_k = G_k^{-1}$,

$$
P_{k+1} = P_k - \frac{P_k a a^\top P_k}{1 + a^\top P_k a} .
$$

The coefficient update is $\theta_{k+1} = \theta_k + P_{k+1}a\,(y - a^\top\theta_k)$.

$$
\boxed{P_{k+1} = P_k - \frac{P_k aa^\top P_k}{1 + a^\top P_k a}, \qquad O(n^2) \text{ per observation}}
$$

**Key takeaway.** The whole point is the cost: refitting from scratch is $O(n^3)$ per new data
point, the update is $O(n^2)$. The same identity is the BFGS inverse-Hessian update and the
Kalman covariance update.

In [30]:
n = 5
X = rng.standard_normal((30, n))
theta_star = rng.standard_normal(n)
y = X @ theta_star + 0.01 * rng.standard_normal(30)
G = X.T @ X
P = np.linalg.inv(G)
theta = P @ X.T @ y

a = rng.standard_normal(n)
ynew = a @ theta_star
P_new = P - np.outer(P @ a, a @ P) / (1 + a @ P @ a)
theta_new = theta + P_new @ a * (ynew - a @ theta)

Xa = np.vstack([X, a])
ya = np.append(y, ynew)
theta_ref = np.linalg.solve(Xa.T @ Xa, Xa.T @ ya)
print("||P_new - (G + a a^T)^-1||_F =", np.linalg.norm(P_new - np.linalg.inv(G + np.outer(a, a))))
print("||theta_new - refit||        =", np.linalg.norm(theta_new - theta_ref))
print("1 + a^T P a =", 1 + a @ P @ a)
assert np.allclose(P_new, np.linalg.inv(G + np.outer(a, a)))
assert np.allclose(theta_new, theta_ref)

||P_new - (G + a a^T)^-1||_F = 5.038169788119109e-17
||theta_new - refit||        = 3.5224712442708127e-16
1 + a^T P a = 1.094976779440053


### Problem L2.8 — Woodbury and the Kalman gain

**Statement.** State and prove the Woodbury identity, then use it to show that the information
form and the covariance form of the Kalman update agree:

$$
\left( P^{-1} + H^\top R^{-1} H \right)^{-1}
= P - PH^\top\left( HPH^\top + R \right)^{-1}HP .
$$

**Intuition.** Woodbury trades an $n \times n$ inverse for a $k \times k$ one; with $k$
measurements and $n$ states, $k \ll n$ is the normal case.

**Solution.**

*Step 1 — the identity.*

$$
(A + UCV)^{-1} = A^{-1} - A^{-1}U\left(C^{-1} + VA^{-1}U\right)^{-1}VA^{-1} .
$$

*Step 2 — proof.* Multiply the right side by $A + UCV$ and expand. The two correction terms
combine as

$$
-UC\left(C^{-1} + VA^{-1}U\right)\left(C^{-1} + VA^{-1}U\right)^{-1}VA^{-1} = -UCVA^{-1},
$$

which cancels the term $UCVA^{-1}$, leaving $I_n$. The same computation on the other side gives
$I_n$ again.

*Step 3 — substitute.* Take $A = P^{-1}$, $U = H^\top$, $C = R^{-1}$ and $V = H$. Then
$A^{-1} = P$ and

$$
\left(P^{-1} + H^\top R^{-1}H\right)^{-1} = P - PH^\top\left(R + HPH^\top\right)^{-1}HP .
$$

*Step 4 — read off the gain.* The matrix $K = PH^\top(HPH^\top + R)^{-1}$ is the **Kalman
gain**, and the posterior covariance is $(I - KH)P$.

$$
\boxed{K = PH^\top\left(HPH^\top + R\right)^{-1}, \qquad P^{+} = (I - KH)P}
$$

**Key takeaway.** The left side inverts an $n \times n$ matrix, the right side a
$k \times k$ one. With $n = 10^3$ states and $k = 3$ sensors, that is the difference between a
billion flops and twenty-seven.

In [31]:
n, k = 6, 2
Gp = rng.standard_normal((n, n))
P = Gp @ Gp.T + n * np.eye(n)
H = rng.standard_normal((k, n))
Gr = rng.standard_normal((k, k))
R = Gr @ Gr.T + k * np.eye(k)

info_form = np.linalg.inv(np.linalg.inv(P) + H.T @ np.linalg.inv(R) @ H)
K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
cov_form = P - K @ H @ P
print("||information form - covariance form||_F =", np.linalg.norm(info_form - cov_form))
print("posterior covariance is SPD:", (np.linalg.eigvalsh(cov_form) > 0).all())
print("uncertainty never increases:",
      bool((np.linalg.eigvalsh(P - cov_form) > -1e-12).all()))
assert np.allclose(info_form, cov_form)
assert (np.linalg.eigvalsh(cov_form) > 0).all()

||information form - covariance form||_F = 3.68438931439377e-15
posterior covariance is SPD: True
uncertainty never increases: True


### Problem L2.9 — Rank-one Cholesky update for a streaming Gaussian process

**Statement.** Let $K = LL^\top$ be a Cholesky-factored kernel matrix and $x \in \mathbb{R}^n$.
Give an $O(n^2)$ algorithm producing $\tilde{L}$ with
$\tilde{L}\tilde{L}^\top = K + xx^\top$, and prove it correct.

**Intuition.** Append $x$ as one extra column and rotate it away; orthogonal rotations do not
change the product.

**Solution.**

*Step 1 — the key observation.* Stack the factors:

$$
K + xx^\top = LL^\top + xx^\top
= \begin{bmatrix} L & x \end{bmatrix}\begin{bmatrix} L^\top \\ x^\top \end{bmatrix} .
$$

*Step 2 — rotate.* Let $Q \in \mathbb{R}^{(n+1) \times (n+1)}$ be a product of $n$ Givens
rotations chosen so that

$$
Q \begin{bmatrix} L^\top \\ x^\top \end{bmatrix} = \begin{bmatrix} \tilde{L}^\top \\ 0 \end{bmatrix},
$$

with $\tilde{L}^\top$ upper triangular. Rotation $k$ acts on row $k$ of $L^\top$ and on the last
row, and zeroes $x_k$.

*Step 3 — conclude.* Transposing the display gives
$\begin{bmatrix} L & x \end{bmatrix}Q^\top = \begin{bmatrix} \tilde{L} & 0 \end{bmatrix}$, so

$$
\tilde{L}\tilde{L}^\top
= \begin{bmatrix} \tilde{L} & 0 \end{bmatrix}\begin{bmatrix} \tilde{L}^\top \\ 0 \end{bmatrix}
= \begin{bmatrix} L & x \end{bmatrix}Q^\top Q \begin{bmatrix} L^\top \\ x^\top \end{bmatrix}
= LL^\top + xx^\top .
$$

Note the vector rotated away is $x$ itself. Substituting $v = L^{-1}x$ first and rotating
$\begin{bmatrix} L^\top \\ v^\top \end{bmatrix}$ would produce $LL^\top + vv^\top$, which is
**not** $K + xx^\top$ unless $L = I$. The correct version of that route is to Cholesky-factor
$I + vv^\top = CC^\top$ and set $\tilde{L} = LC$, since
$K + xx^\top = L(I + vv^\top)L^\top$.

*Step 4 — cost.* Each of the $n$ rotations touches $O(n)$ entries, so the update is $O(n^2)$
against $\tfrac13 n^3$ for a fresh factorization.

$$
\boxed{\tilde{L}\tilde{L}^\top = K + xx^\top \text{ in } O(n^2) \text{ flops, via } n \text{ Givens rotations applied to } x \text{ (not to } L^{-1}x)}
$$

**Key takeaway.** Both routes are correct once stated correctly, and both are $O(n^2)$; the
rotation form is the one used in practice because it never forms $L^{-1}$.

In [32]:
def cholupdate(L, x):
    """Rank-one Cholesky update: returns Lt with Lt Lt^T = L L^T + x x^T."""
    L, x = np.array(L, dtype=float), np.array(x, dtype=float)
    n = len(x)
    for k in range(n):
        r = np.hypot(L[k, k], x[k])
        c, s = r / L[k, k], x[k] / L[k, k]
        L[k, k] = r
        if k + 1 < n:
            L[k + 1:, k] = (L[k + 1:, k] + s * x[k + 1:]) / c
            x[k + 1:] = c * x[k + 1:] - s * L[k + 1:, k]
    return L


n = 6
Gk = rng.standard_normal((n, n))
K = Gk @ Gk.T + n * np.eye(n)
L = np.linalg.cholesky(K)
xv = rng.standard_normal(n)

Lt = cholupdate(L, xv)
L_ref = np.linalg.cholesky(K + np.outer(xv, xv))
print("||Lt Lt^T - (K + x x^T)||_F =", np.linalg.norm(Lt @ Lt.T - (K + np.outer(xv, xv))))
print("||Lt - fresh cholesky||_F   =", np.linalg.norm(Lt - L_ref))

vv = sla.solve_triangular(L, xv, lower=True)
C = np.linalg.cholesky(np.eye(n) + np.outer(vv, vv))
L_alt = L @ C
print("alternative route L C:  ||L C (L C)^T - (K + x x^T)||_F =",
      np.linalg.norm(L_alt @ L_alt.T - (K + np.outer(xv, xv))))
print("the discarded v-substitution would give K + v v^T, wrong by",
      np.linalg.norm(np.outer(vv, vv) - np.outer(xv, xv)), "in Frobenius norm")
assert np.allclose(Lt @ Lt.T, K + np.outer(xv, xv))
assert np.allclose(Lt, L_ref)
assert np.allclose(L_alt @ L_alt.T, K + np.outer(xv, xv))

||Lt Lt^T - (K + x x^T)||_F = 3.1056492334922947e-15
||Lt - fresh cholesky||_F   = 5.617076402713773e-16
alternative route L C:  ||L C (L C)^T - (K + x x^T)||_F = 3.049578743934494e-15
the discarded v-substitution would give K + v v^T, wrong by 2.5438789410374514 in Frobenius norm


### Problem L2.10 — Gaussian process posterior through the Cholesky factor

**Statement.** In Gaussian process regression with $K_\sigma = K + \sigma^2 I$, derive the
algorithm computing

$$
\bar{f}_{\star} = k_{\star}^\top K_\sigma^{-1}y,
\qquad
\operatorname{Var}(f_{\star}) = k(x_{\star},x_{\star}) - k_{\star}^\top K_\sigma^{-1}k_{\star},
$$

using only triangular solves with $K_\sigma = LL^\top$, and give the log marginal likelihood.

**Intuition.** Every occurrence of $K_\sigma^{-1}$ is a solve, and the variance is a squared
norm once $L^{-1}$ has been applied.

**Solution.**

*Step 1 — factor.* $K_\sigma$ is SPD because $K \succeq 0$ and $\sigma^2 \gt 0$, so
Theorem 4.6 gives $K_\sigma = LL^\top$ at $\tfrac13 n^3$ flops.

*Step 2 — the mean.* Solve $L\alpha = y$ then $L^\top\beta = \alpha$. Then
$\beta = (LL^\top)^{-1}y = K_\sigma^{-1}y$, and $\bar{f}_{\star} = k_{\star}^\top\beta$.

*Step 3 — the variance.* Solve $Lv = k_{\star}$. Then

$$
k_{\star}^\top K_\sigma^{-1}k_{\star} = k_{\star}^\top L^{-\top}L^{-1}k_{\star}
= (L^{-1}k_{\star})^\top(L^{-1}k_{\star}) = \lVert v \rVert_2^2 .
$$

*Step 4 — the log determinant.* $\det K_\sigma = (\det L)^2 = \prod_i l_{ii}^2$, so

$$
\log p(y) = -\tfrac12 y^\top\beta - \sum_i \log l_{ii} - \tfrac{n}{2}\log 2\pi .
$$

$$
\boxed{\bar{f}_{\star} = k_{\star}^\top\beta, \quad
\operatorname{Var}(f_{\star}) = k(x_{\star},x_{\star}) - \lVert v \rVert_2^2, \quad
\log\lvert K_\sigma \rvert = 2\sum_i \log l_{ii}}
$$

**Key takeaway.** The variance formula is manifestly non-negative once written as
$k_{\star\star} - \lVert v \rVert^2$ with $v$ from a triangular solve — an explicit-inverse
implementation can return a small negative variance through rounding, which this one cannot for
any $v$ with $\lVert v \rVert^2 \le k_{\star\star}$.

In [33]:
xs = np.linspace(0.0, 1.0, 8)
ys = np.sin(2 * np.pi * xs) + 0.05 * rng.standard_normal(8)
ell, sigma2 = 0.2, 1e-2
Kmat = np.exp(-0.5 * (xs[:, None] - xs[None, :]) ** 2 / ell ** 2)
Ksig = Kmat + sigma2 * np.eye(8)
L = np.linalg.cholesky(Ksig)
alpha = sla.solve_triangular(L, ys, lower=True)
beta = sla.solve_triangular(L.T, alpha, lower=False)
for xstar in (0.13, 0.5, 0.91):
    kstar = np.exp(-0.5 * (xs - xstar) ** 2 / ell ** 2)
    v = sla.solve_triangular(L, kstar, lower=True)
    mean_c, var_c = kstar @ beta, 1.0 - v @ v
    Kinv = np.linalg.inv(Ksig)
    mean_i, var_i = kstar @ Kinv @ ys, 1.0 - kstar @ Kinv @ kstar
    print(f"x* = {xstar:.2f}   mean {mean_c: .10f} vs {mean_i: .10f}"
          f"   var {var_c:.10f} vs {var_i:.10f}")
    assert abs(mean_c - mean_i) < 1e-9 and abs(var_c - var_i) < 1e-9
    assert var_c > 0
logdet = 2 * np.log(np.diag(L)).sum()
loglik = -0.5 * ys @ beta - np.log(np.diag(L)).sum() - 4 * np.log(2 * np.pi)
print(f"log|K_sig| = {logdet:.10f}   slogdet = {np.linalg.slogdet(Ksig)[1]:.10f}")
print(f"log marginal likelihood = {loglik:.10f}")
assert np.isclose(logdet, np.linalg.slogdet(Ksig)[1])

x* = 0.13   mean  0.6550416993 vs  0.6550416993   var 0.0089686565 vs 0.0089686565
x* = 0.50   mean -0.0182351493 vs -0.0182351493   var 0.0077390678 vs 0.0077390678
x* = 0.91   mean -0.5080684387 vs -0.5080684387   var 0.0096287443 vs 0.0096287443
log|K_sig| = -9.3304967668   slogdet = -9.3304967668
log marginal likelihood = -4.3150469244


## L3 — Challenge Proofs

### Problem L3.1 — Cholesky preserves bandwidth

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ be SPD with lower bandwidth $p$, meaning
$a_{ij} = 0$ whenever $i - j \gt p$. Prove that its Cholesky factor satisfies $l_{ij} = 0$
whenever $i - j \gt p$.

**Intuition.** Elimination can only create a nonzero where two existing nonzeros meet, and
outside the band they never do.

**Solution.** Induct on the column index $k$, with the inductive hypothesis that
$l_{ij} = 0$ for all $j \lt k$ with $i - j \gt p$.

*Step 1 — the formula.* For $i \gt k$,

$$
l_{ik} = \frac{1}{l_{kk}}\left( a_{ik} - \sum_{j=1}^{k-1} l_{ij}l_{kj} \right) .
$$

*Step 2 — the direct term.* Suppose $i - k \gt p$. Bandwidth of $A$ gives $a_{ik} = 0$.

*Step 3 — every product vanishes.* Fix $j \lt k$ and split on the size of $j$.

If $j \lt k - p$ then $k - j \gt p$, so $l_{kj} = 0$ by the inductive hypothesis.

If $j \ge k - p$ then $i - j \ge i - k \gt p$, since $j \le k$; so $l_{ij} = 0$, again by the
inductive hypothesis.

*Step 4 — conclude.* Every term of the sum is zero and $a_{ik} = 0$, so $l_{ik} = 0$. The
hypothesis now holds for column $k$, closing the induction.

$$
\boxed{\text{bandwidth}(L) \le p, \quad \text{so Cholesky costs } O(np^2) \text{ flops and } O(np) \text{ storage}}
$$

**Key takeaway.** For a tridiagonal matrix $p = 1$, and the factorization is $O(n)$: that is the
Thomas algorithm of Problem L2.1. For a general sparse matrix no such bound holds — elimination
creates **fill-in**, which is why reorderings such as minimum degree and nested dissection are
applied first.

In [34]:
def banded_spd(n, p, rng):
    B = np.triu(np.tril(rng.standard_normal((n, n)), p), -p)
    return B @ B.T + 3 * p * n * np.eye(n)


def bandwidth(M, tol=1e-10):
    scale = max(np.abs(M).max(), 1.0)
    idx = [abs(i - j) for i in range(M.shape[0]) for j in range(M.shape[1])
           if abs(M[i, j]) > tol * scale]
    return int(max(idx))


for p in (1, 2, 4):
    A = banded_spd(40, p, rng)
    A = np.triu(np.tril(A, p), -p)
    A = A + (abs(np.linalg.eigvalsh(A)[0]) + 1.0) * np.eye(40)
    L = np.linalg.cholesky(A)
    print(f"bandwidth(A) = {bandwidth(A)}   bandwidth(L) = {bandwidth(L)}"
          f"   (bound {p})   ||A - L L^T||_F = {np.linalg.norm(A - L @ L.T):.2e}")
    assert bandwidth(L) <= p

Adense = rng.standard_normal((40, 40))
Adense = Adense @ Adense.T + 40 * np.eye(40)
print("a dense SPD matrix for contrast: bandwidth(L) =",
      bandwidth(np.linalg.cholesky(Adense)))

bandwidth(A) = 1   bandwidth(L) = 1   (bound 1)   ||A - L L^T||_F = 1.33e-13
bandwidth(A) = 2   bandwidth(L) = 2   (bound 2)   ||A - L L^T||_F = 3.41e-13
bandwidth(A) = 4   bandwidth(L) = 4   (bound 4)   ||A - L L^T||_F = 4.82e-13
a dense SPD matrix for contrast: bandwidth(L) = 39


### Problem L3.2 — Block LU and the Schur complement

**Statement.** For $M = \left[\begin{smallmatrix}A & B \\ C & D\end{smallmatrix}\right]$ with
$A$ nonsingular, factor $M$ into block lower triangular, block diagonal and block upper
triangular factors.

**Intuition.** One block elimination step clears $C$, and the leftover in the corner is the
Schur complement.

**Solution.**

*Step 1 — eliminate $C$.* Left-multiply by
$E = \left[\begin{smallmatrix}I & 0 \\ -CA^{-1} & I\end{smallmatrix}\right]$:

$$
EM = \begin{bmatrix} A & B \\ 0 & D - CA^{-1}B \end{bmatrix} .
$$

*Step 2 — name the corner.* $S = D - CA^{-1}B$.

*Step 3 — undo the elimination.* $E^{-1} = \left[\begin{smallmatrix}I & 0 \\ CA^{-1} & I\end{smallmatrix}\right]$,
so $M = E^{-1}\left[\begin{smallmatrix}A & B \\ 0 & S\end{smallmatrix}\right]$.

*Step 4 — split the remaining factor.* Clearing $B$ on the right,

$$
\begin{bmatrix} A & B \\ 0 & S \end{bmatrix}
= \begin{bmatrix} A & 0 \\ 0 & S \end{bmatrix}
\begin{bmatrix} I & A^{-1}B \\ 0 & I \end{bmatrix} .
$$

$$
\boxed{M = \begin{bmatrix} I & 0 \\ CA^{-1} & I \end{bmatrix}
\begin{bmatrix} A & 0 \\ 0 & D - CA^{-1}B \end{bmatrix}
\begin{bmatrix} I & A^{-1}B \\ 0 & I \end{bmatrix}}
$$

**Key takeaway.** This is Theorem 4.8. Scalar Gaussian elimination is the special case in which
$A$ is the $1 \times 1$ pivot, so every elimination step is a Schur complement.

In [35]:
p, q = 3, 2
M = rng.standard_normal((p + q, p + q))
A_, B_ = M[:p, :p], M[:p, p:]
C_, D_ = M[p:, :p], M[p:, p:]
S_ = D_ - C_ @ np.linalg.inv(A_) @ B_
Lb = np.block([[np.eye(p), np.zeros((p, q))], [C_ @ np.linalg.inv(A_), np.eye(q)]])
Db = np.block([[A_, np.zeros((p, q))], [np.zeros((q, p)), S_]])
Ub = np.block([[np.eye(p), np.linalg.inv(A_) @ B_], [np.zeros((q, p)), np.eye(q)]])
print("||M - L D U||_F =", np.linalg.norm(M - Lb @ Db @ Ub))
print("det M =", np.linalg.det(M), "   det A * det S =",
      np.linalg.det(A_) * np.linalg.det(S_))
assert np.allclose(M, Lb @ Db @ Ub)
assert np.isclose(np.linalg.det(M), np.linalg.det(A_) * np.linalg.det(S_))

||M - L D U||_F = 1.2241249243000656e-15
det M = -6.148369670609989    det A * det S = -6.148369670609988


### Problem L3.3 — The inverse of a block triangular matrix

**Statement.** Compute $T^{-1}$ for
$T = \left[\begin{smallmatrix}A & B \\ 0 & C\end{smallmatrix}\right]$ with $A$ and $C$
nonsingular.

**Intuition.** Triangularity survives inversion, so the $(2,1)$ block of the inverse must
vanish.

**Solution.** Write $T^{-1} = \left[\begin{smallmatrix}W & X \\ Y & Z\end{smallmatrix}\right]$
and expand $TT^{-1} = I$.

*Step 1 — block $(2,1)$.* $CY = 0$ and $C$ nonsingular give $Y = 0$.

*Step 2 — block $(2,2)$.* $CZ = I$ gives $Z = C^{-1}$.

*Step 3 — block $(1,1)$.* $AW + BY = AW = I$ gives $W = A^{-1}$.

*Step 4 — block $(1,2)$.* $AX + BZ = 0$ gives $X = -A^{-1}BC^{-1}$.

$$
\boxed{T^{-1} = \begin{bmatrix} A^{-1} & -A^{-1}BC^{-1} \\ 0 & C^{-1} \end{bmatrix}}
$$

**Key takeaway.** Only the diagonal blocks are inverted; the coupling block costs two
multiplications. This is what makes block back substitution work, and it is the block version
of Problem L1.8.

In [36]:
p, q = 3, 2
A_ = rng.standard_normal((p, p)) + p * np.eye(p)
B_ = rng.standard_normal((p, q))
C_ = rng.standard_normal((q, q)) + q * np.eye(q)
T = np.block([[A_, B_], [np.zeros((q, p)), C_]])
Tinv = np.block([[np.linalg.inv(A_), -np.linalg.inv(A_) @ B_ @ np.linalg.inv(C_)],
                 [np.zeros((q, p)), np.linalg.inv(C_)]])
print("||T Tinv - I||_F =", np.linalg.norm(T @ Tinv - np.eye(p + q)))
print("||Tinv - numpy inverse||_F =", np.linalg.norm(Tinv - np.linalg.inv(T)))
assert np.allclose(T @ Tinv, np.eye(p + q))

||T Tinv - I||_F = 4.4225605943810797e-16
||Tinv - numpy inverse||_F = 6.283435673255384e-17


### Problem L3.4 — The Schur determinant identity

**Statement.** For $M = \left[\begin{smallmatrix}A & B \\ C & D\end{smallmatrix}\right]$ with
$A$ nonsingular, prove $\det M = \det A \cdot \det(D - CA^{-1}B)$.

**Intuition.** Block triangularization turns a determinant of a big matrix into a product of
determinants of small ones.

**Solution.** Two standard properties of the determinant are used: multiplicativity, and that
the determinant of a block triangular matrix is the product of the determinants of its diagonal
blocks.

*Step 1 — factor.* By Problem L3.2,

$$
M = \begin{bmatrix} I & 0 \\ CA^{-1} & I \end{bmatrix}
\begin{bmatrix} A & B \\ 0 & S \end{bmatrix},
\qquad S = D - CA^{-1}B .
$$

*Step 2 — the first factor.* It is block lower triangular with identity diagonal blocks, so its
determinant is $\det I \cdot \det I = 1$.

*Step 3 — the second factor.* It is block upper triangular, so its determinant is
$\det A \cdot \det S$.

*Step 4 — multiply.*

$$
\boxed{\det M = \det A \cdot \det\left(D - CA^{-1}B\right)}
$$

**Key takeaway.** With $A = a_{11}$ this says $\det M = a_{11}\det S$, and iterating gives
$\det M = \prod_k$ pivots — the same formula as Problem L1.7 read one step at a time.

In [37]:
for (p, q) in [(1, 3), (2, 2), (3, 1)]:
    M = rng.standard_normal((p + q, p + q))
    A_, B_ = M[:p, :p], M[:p, p:]
    C_, D_ = M[p:, :p], M[p:, p:]
    S_ = D_ - C_ @ np.linalg.inv(A_) @ B_
    lhs, rhs = np.linalg.det(M), np.linalg.det(A_) * np.linalg.det(S_)
    print(f"p={p} q={q}:  det M = {lhs: .10f}   det A * det S = {rhs: .10f}"
          f"   difference = {abs(lhs - rhs):.2e}")
    assert np.isclose(lhs, rhs)

p=1 q=3:  det M = -2.6748453148   det A * det S = -2.6748453148   difference = 0.00e+00
p=2 q=2:  det M =  15.7655187839   det A * det S =  15.7655187839   difference = 0.00e+00
p=3 q=1:  det M =  9.1626009738   det A * det S =  9.1626009738   difference = 1.78e-15


### Problem L3.5 — The growth factor bound $2^{n-1}$, and a matrix that attains it

**Statement.** Prove that Gaussian elimination with partial pivoting satisfies $\rho \le 2^{n-1}$,
and exhibit a $4 \times 4$ matrix with $\rho = 8$.

**Intuition.** Each elimination step can at most double the largest entry, and there are $n-1$
steps.

**Solution.**

*Step 1 — one step doubles at most.* At step $k$, entries update as
$a_{ij}^{(k)} = a_{ij}^{(k-1)} - l_{ik}a_{kj}^{(k-1)}$. Partial pivoting gives
$\lvert l_{ik} \rvert \le 1$, so

$$
\lvert a_{ij}^{(k)} \rvert \le \lvert a_{ij}^{(k-1)} \rvert + \lvert a_{kj}^{(k-1)} \rvert
\le 2 \max_{m,q}\lvert a_{mq}^{(k-1)} \rvert .
$$

*Step 2 — induct over $n-1$ steps.*

$$
\max_{i,j,k}\lvert a_{ij}^{(k)} \rvert \le 2^{n-1}\max_{i,j}\lvert a_{ij} \rvert,
\qquad \text{so } \rho \le 2^{n-1} .
$$

*Step 3 — attain it.* Take the Wilkinson matrix

$$
W_4 = \begin{bmatrix}
1 & 0 & 0 & 1 \\ -1 & 1 & 0 & 1 \\ -1 & -1 & 1 & 1 \\ -1 & -1 & -1 & 1
\end{bmatrix} .
$$

Every column-$1$ entry has modulus $1$, so partial pivoting performs no swap and the
multipliers are all $-1$. Step $1$ turns the last column into $(1,2,2,2)$; step $2$ into
$(1,2,4,4)$; step $3$ into $(1,2,4,8)$.

*Step 4 — read off $\rho$.* The largest entry ever produced is $8$ and the largest initial
entry is $1$.

$$
\boxed{\rho \le 2^{n-1}, \text{ attained: } \rho(W_4) = 8 = 2^3}
$$

**Key takeaway.** The bound is sharp, so partial pivoting is not unconditionally stable in the
worst case. What saves it is that $\rho$ is tiny for practically every matrix that is not built
to break it: the cell below and Section 7.5 of the theory notebook both measure a median
growth under $2$ at $n = 14$, against a worst case of $8192$.

In [38]:
def growth_factor(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    U, biggest = A.copy(), np.abs(A).max()
    for k in range(n - 1):
        p = k + int(np.argmax(np.abs(U[k:, k])))
        if p != k:
            U[[k, p], :] = U[[p, k], :]
        if U[k, k] == 0.0:
            continue
        m = U[k + 1:, k] / U[k, k]
        U[k + 1:, k:] -= np.outer(m, U[k, k:])
        biggest = max(biggest, np.abs(U).max())
    return biggest / np.abs(A).max()


W4 = np.array([[1.0, 0.0, 0.0, 1.0],
               [-1.0, 1.0, 0.0, 1.0],
               [-1.0, -1.0, 1.0, 1.0],
               [-1.0, -1.0, -1.0, 1.0]])
print("rho(W4) =", growth_factor(W4), "   2^3 =", 8)
for n in (6, 10, 14):
    Wn = -np.tril(np.ones((n, n)), -1) + np.eye(n)
    Wn[:, -1] = 1.0
    print(f"n = {n:2d}:  rho = {growth_factor(Wn):8.0f}   2^(n-1) = {2.0 ** (n - 1):8.0f}"
          f"   median random rho = "
          f"{np.median([growth_factor(rng.standard_normal((n, n))) for _ in range(40)]):.3f}")
    assert np.isclose(growth_factor(Wn), 2.0 ** (n - 1))
assert np.isclose(growth_factor(W4), 8.0)

rho(W4) = 8.0    2^3 = 8
n =  6:  rho =       32   2^(n-1) =       32   median random rho = 1.181
n = 10:  rho =      512   2^(n-1) =      512   median random rho = 1.481
n = 14:  rho =     8192   2^(n-1) =     8192   median random rho = 1.592


### Problem L3.6 — Uniqueness of the LU factorization

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ be nonsingular and suppose $A = L_1U_1 = L_2U_2$
with $L_1, L_2$ unit lower triangular and $U_1, U_2$ upper triangular. Prove $L_1 = L_2$ and
$U_1 = U_2$.

**Intuition.** Rearranging turns the two factorizations into a single matrix that is both lower
and upper triangular.

**Solution.**

*Step 1 — everything is invertible.* $\det A \neq 0$ forces $\det U_i \neq 0$, and each $L_i$
has determinant $1$.

*Step 2 — rearrange.* From $L_1U_1 = L_2U_2$,

$$
L_2^{-1}L_1 = U_2U_1^{-1} .
$$

*Step 3 — classify the common value.* The inverse of a unit lower triangular matrix is unit
lower triangular, and a product of two such is unit lower triangular, so the left side is unit
lower triangular. The right side is upper triangular for the same reason.

*Step 4 — conclude.* A matrix that is both lower and upper triangular is diagonal, and its
diagonal is that of the left side, namely all ones. So both sides equal $I$.

$$
\boxed{L_1 = L_2 \quad \text{and} \quad U_1 = U_2}
$$

**Key takeaway.** Uniqueness needs the normalization $l_{ii} = 1$: without it $A = (LD)(D^{-1}U)$
for any nonsingular diagonal $D$, so the factorization is unique only up to a diagonal scaling.
It is also uniqueness *for a fixed row order*: a different permutation $P$ in Theorem 4.5 gives
a genuinely different pair of factors.

In [39]:
def lu_no_pivot(A):
    """Doolittle LU without row interchanges."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L, U = np.eye(n), A.copy()
    for k in range(n - 1):
        L[k + 1:, k] = U[k + 1:, k] / U[k, k]
        U[k + 1:, k:] -= np.outer(L[k + 1:, k], U[k, k:])
        U[k + 1:, k] = 0.0
    return L, U


A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L1, U1 = lu_no_pivot(A)
L2 = np.array([[1.0, 0.0, 0.0], [2.0, 1.0, 0.0], [4.0, 3.0, 1.0]])
U2 = np.array([[2.0, 1.0, 1.0], [0.0, 1.0, 1.0], [0.0, 0.0, 2.0]])
print("elimination gives L =\n", L1, "\nU =\n", U1)
print("||L1 - L2||_F =", np.linalg.norm(L1 - L2), "   ||U1 - U2||_F =", np.linalg.norm(U1 - U2))
D = np.diag([3.0, -2.0, 0.5])
print("without the unit-diagonal normalization, (L D)(D^-1 U) also works:",
      np.allclose((L2 @ D) @ (np.linalg.inv(D) @ U2), A))
Pp, Lp, Up = sla.lu(A)
print("a different permutation gives a different pair: P = I?", np.allclose(Pp, np.eye(3)),
      "  ||P^T A - Lp Up|| =", np.linalg.norm(Pp.T @ A - Lp @ Up))
assert np.allclose(L1, L2) and np.allclose(U1, U2)
assert np.allclose((L2 @ D) @ (np.linalg.inv(D) @ U2), A)
assert not np.allclose(Lp, L2)

elimination gives L =
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]] 
U =
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
||L1 - L2||_F = 0.0    ||U1 - U2||_F = 0.0
without the unit-diagonal normalization, (L D)(D^-1 U) also works: True
a different permutation gives a different pair: P = I? False   ||P^T A - Lp Up|| = 4.440892098500626e-16


### Problem L3.7 — The rigorous perturbation theorem

**Statement.** Let $Ax = b$ and $(A + \delta A)(x + \delta x) = b + \delta b$ with
$\lVert A^{-1} \rVert \lVert \delta A \rVert \lt 1$. Prove

$$
\frac{\lVert \delta x \rVert}{\lVert x \rVert}
\le \frac{\kappa(A)}{1 - \kappa(A)\lVert \delta A \rVert / \lVert A \rVert}
\left( \frac{\lVert \delta b \rVert}{\lVert b \rVert} + \frac{\lVert \delta A \rVert}{\lVert A \rVert} \right) .
$$

**Intuition.** The denominator is the price of perturbing $A$ itself: it says the bound
degrades as $A + \delta A$ approaches singularity.

**Solution.**

*Step 1 — an exact equation.* Expanding and subtracting $Ax = b$,

$$
(A + \delta A)\delta x = \delta b - \delta A\,x .
$$

No second-order term is dropped: $\delta A\,\delta x$ sits inside the left-hand side.

*Step 2 — invert.* Write $A + \delta A = A(I + A^{-1}\delta A)$. Since
$\lVert A^{-1}\delta A \rVert \le \lVert A^{-1} \rVert \lVert \delta A \rVert \lt 1$, the
Neumann series gives invertibility and

$$
\lVert (I + A^{-1}\delta A)^{-1} \rVert \le \frac{1}{1 - \lVert A^{-1} \rVert \lVert \delta A \rVert} .
$$

*Step 3 — take norms.*

$$
\lVert \delta x \rVert \le \frac{\lVert A^{-1} \rVert}{1 - \lVert A^{-1} \rVert \lVert \delta A \rVert}
\left( \lVert \delta b \rVert + \lVert \delta A \rVert \lVert x \rVert \right) .
$$

*Step 4 — relativize.* Divide by $\lVert x \rVert$, use
$1/\lVert x \rVert \le \lVert A \rVert/\lVert b \rVert$, and multiply numerator and denominator
by $\lVert A \rVert$.

$$
\boxed{\frac{\lVert \delta x \rVert}{\lVert x \rVert}
\le \frac{\kappa(A)}{1 - \kappa(A)\frac{\lVert \delta A \rVert}{\lVert A \rVert}}
\left( \frac{\lVert \delta b \rVert}{\lVert b \rVert} + \frac{\lVert \delta A \rVert}{\lVert A \rVert} \right)}
$$

**Key takeaway.** This is Theorem 4.10. The hypothesis
$\kappa(A)\lVert \delta A \rVert/\lVert A \rVert \lt 1$ is exactly the statement that the
perturbation is smaller than the distance from $A$ to the nearest singular matrix, which in the
$2$-norm is $\sigma_n(A)$.

In [40]:
A = np.array([[1.0, 2.0], [1.0, 2.01]])
b = np.array([3.0, 3.01])
x = np.linalg.solve(A, b)
kap = np.linalg.cond(A, np.inf)
nA, nb = np.linalg.norm(A, np.inf), np.linalg.norm(b, np.inf)
print("kappa_inf =", kap, "  sigma_min(A) =", np.linalg.svd(A, compute_uv=False)[-1])
worst = 0.0
for _ in range(300):
    dA = 1e-6 * rng.standard_normal((2, 2))
    db = 1e-6 * rng.standard_normal(2)
    if kap * np.linalg.norm(dA, np.inf) / nA >= 1:
        continue
    dx = np.linalg.solve(A + dA, b + db) - x
    lhs = np.linalg.norm(dx, np.inf) / np.linalg.norm(x, np.inf)
    rhs = (kap / (1 - kap * np.linalg.norm(dA, np.inf) / nA)) * (
        np.linalg.norm(db, np.inf) / nb + np.linalg.norm(dA, np.inf) / nA)
    worst = max(worst, lhs / rhs)
    assert lhs <= rhs
print(f"300 random perturbations: bound never violated; tightest lhs/rhs = {worst:.4f}")

kappa_inf = 1207.0100000000257   sigma_min(A) = 0.0031559578640154287
300 random perturbations: bound never violated; tightest lhs/rhs = 0.9666


### Problem L3.8 — Inertia from $LDL^\top$, and Sylvester's law

**Statement.** State Sylvester's law of inertia and explain how the diagonal of $D$ in
$A = LDL^\top$ reveals the inertia $(n_+, n_-, n_0)$ of a symmetric $A$. State precisely when
such a factorization exists.

**Intuition.** Congruence can rescale and mix directions but cannot turn a direction of
positive curvature into one of negative curvature.

**Solution.**

*Step 1 — the law.* If $A$ and $B$ are real symmetric and congruent, $A = SBS^\top$ with $S$
nonsingular, then $A$ and $B$ have the same inertia.

*Step 2 — apply it.* $L$ is unit lower triangular, so $\det L = 1$ and $L$ is nonsingular.
Hence $A = LDL^\top$ makes $A$ congruent to $D$.

*Step 3 — read off the counts.* $D$ is diagonal, so its eigenvalues are its diagonal entries,
and

$$
n_+ = \#\lbrace i : d_i \gt 0 \rbrace, \quad
n_- = \#\lbrace i : d_i \lt 0 \rbrace, \quad
n_0 = \#\lbrace i : d_i = 0 \rbrace .
$$

*Step 4 — the missing hypothesis.* A diagonal $D$ does not always exist. The factorization
$A = LDL^\top$ with $L$ unit lower triangular and $D$ diagonal exists exactly when every leading
principal submatrix $A_1, \dots, A_{n-1}$ is nonsingular — the same condition as Theorem 4.4,
since $LDL^\top$ is the LU factorization with $U = DL^\top$.

The matrix $A = \left[\begin{smallmatrix}0 & 1 \\ 1 & 0\end{smallmatrix}\right]$ has $A_1 = [0]$
singular and admits no such factorization: $d_1 = 0$ would force $l_{21}d_1 = 1$.

*Step 5 — the general case.* Bunch-Kaufman pivoting produces $PAP^\top = LDL^\top$ with $D$
block diagonal, using $1 \times 1$ and $2 \times 2$ blocks. A $2 \times 2$ block contributes
one positive and one negative eigenvalue, so the inertia is still read off the blocks.

$$
\boxed{\operatorname{inertia}(A) = \operatorname{inertia}(D), \text{ provided } A_1, \dots, A_{n-1} \text{ are nonsingular; otherwise use Bunch-Kaufman}}
$$

**Key takeaway.** Inertia costs one $\tfrac13 n^3$ factorization, not an eigenvalue
decomposition. This is how interior-point solvers check second-order conditions on a KKT matrix
at every iteration.

In [41]:
def ldl_no_pivot(A):
    """A = L D L^T without pivoting; raises if a leading submatrix is singular."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L, d = np.eye(n), np.zeros(n)
    for k in range(n):
        d[k] = A[k, k] - (L[k, :k] ** 2 * d[:k]).sum()
        if abs(d[k]) < 1e-14 and k < n - 1:
            raise ValueError(f"pivot {k} is {d[k]:g}: leading submatrix singular")
        if k + 1 < n:
            L[k + 1:, k] = (A[k + 1:, k] - L[k + 1:, :k] @ (L[k, :k] * d[:k])) / d[k]
    return L, d


for name, A in [("SPD      ", np.array([[4.0, 12.0], [12.0, 45.0]])),
                ("indefinite", np.array([[1.0, 2.0, 3.0], [2.0, 1.0, 1.0], [3.0, 1.0, 0.0]]))]:
    L, d = ldl_no_pivot(A)
    ev = np.linalg.eigvalsh(A)
    inertia_d = (int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum()))
    inertia_ev = (int((ev > 1e-12).sum()), int((ev < -1e-12).sum()),
                  int((np.abs(ev) <= 1e-12).sum()))
    print(f"{name}: pivots = {d}   inertia from D = {inertia_d}"
          f"   from eigenvalues = {inertia_ev}   ||A - L D L^T|| = "
          f"{np.linalg.norm(A - L @ np.diag(d) @ L.T):.2e}")
    assert inertia_d == inertia_ev
    assert np.allclose(A, L @ np.diag(d) @ L.T)

try:
    ldl_no_pivot(np.array([[0.0, 1.0], [1.0, 0.0]]))
except ValueError as err:
    print("no LDL^T for [[0,1],[1,0]]:", err)
lu_b, d_b, perm_b = sla.ldl(np.array([[0.0, 1.0], [1.0, 0.0]]))
ev0 = np.linalg.eigvalsh(np.array([[0.0, 1.0], [1.0, 0.0]]))
print("Bunch-Kaufman D (a single 2x2 block):\n", d_b)
print("its eigenvalues", np.linalg.eigvalsh(d_b), "match those of A", ev0)
assert np.allclose(np.sort(np.linalg.eigvalsh(d_b)), np.sort(ev0))

SPD      : pivots = [4. 9.]   inertia from D = (2, 0, 0)   from eigenvalues = (2, 0, 0)   ||A - L D L^T|| = 0.00e+00
indefinite: pivots = [ 1.     -3.     -0.6667]   inertia from D = (1, 2, 0)   from eigenvalues = (1, 2, 0)   ||A - L D L^T|| = 3.14e-16
no LDL^T for [[0,1],[1,0]]: pivot 0 is 0: leading submatrix singular
Bunch-Kaufman D (a single 2x2 block):
 [[0. 1.]
 [1. 0.]]
its eigenvalues [-1.  1.] match those of A [-1.  1.]
